# Hoppe Additivity Analysis
### Perovskite and Pyrochlore Screening from MaplePy Outputs

**Purpose:** Compute Hoppe additivity residuals for ternary oxides
processed by MaplePy, classify failures, and generate publication
figures for *J. Appl. Cryst.*

**Relationship to MaplePy:** This notebook is a downstream consumer.
It reads `maple_summary.csv` and `maple_results.csv` produced by
MaplePy’s batch processing (Section 5 or 7), and does not modify
or extend the MaplePy codebase.

**Requirements:** pandas, numpy, matplotlib.  No pymatgen needed
(all crystallographic computation was done upstream by MaplePy).

---

In [ ]:
# !pip install adjustText

## 0. Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

# ── Sheffield colour palette ───────────────────────────────────────────────────
SHEFFIELD = {
    'brand_violet' : '#440099',
    'dark_violet'  : '#24125E',
    'electric'     : '#7000FF',
    'powder_violet': '#A78CFA',
    'pale_violet'  : '#C8BBFF',
    'teal'         : '#005A8F',
    'aqua'         : '#00BBCC',
    'coral'        : '#E7004C',
    'flamingo'     : '#FF6371',
    'peach'        : '#FF9664',
    'peak_green'   : '#005750',
    'spearmint'    : '#3BD4AE',
    'midnight'     : '#131E29',
    'pastel_green' : '#A1DED2',
    'powder_blue'  : '#9ADBE8',
}

# ── The version, named once ────────────────────────────────────────────────
# v1.2.1: written into the classified export as a column, so a CSV on disk
# says which notebook classified it. MaplePy does the same.
# v1.2.9: the decomposition records which binaries are estimates; see
# ESTIMATED_BINARIES in Section 1.
HOPPER_VERSION = '1.2.9'

# ── Hoppe's criterion, named once ──────────────────────────────────────────
#
# v1.2.1: 1.0 was the default argument of classify_additivity, the axvline in
# Figure 1, the "< 1%" in the figure's annotation and the "GOOD (<1%)" in the
# decomposition print. Four literals for one criterion, and the figure and the
# classifier were counting different populations against it: 662 against 661
# on the perovskite run, both in the paper.
ADDITIVE_THRESHOLD_PCT = 1.0   # Hoppe's additivity criterion, per cent

# ── When a mean oxidation state is not a state ─────────────────────────────
#
# v1.2.3: _binary_formula_for rounds abs(oxi_state) to pick a binary, and said
# nothing about having done it. On a site shared by two cations the
# occupancy-weighted mean is not an oxidation state at all: Y3NbO7 has Y and
# Nb on one site at a mean of 3.5, rounded up to 4, and the decomposition then
# asked for YO2. Yttrium has no tetravalent oxide because yttrium has no
# tetravalent state.
#
# v1.2.6: the measurement that used to be here was taken over the 30
# structures with a missing binary and claimed an empty band between 0.05 and
# 0.45 of an integer. Re-measured over ALL 29,226 sites in the two families
# there is no empty band anywhere:
#
#     exactly integral   28,218       0.15 to 0.25     136
#     0.002 to 0.05         408       0.25 to 0.35      64
#     0.05  to 0.15         344       0.35 to 0.45      32
#                                     0.45 to 0.50      24
#
# So this is a JUDGEMENT, not a measured gap, and moving it moves the 1.4
# population: 408 sites lie inside it and 600 outside. It is small enough
# that a genuine half-integer mean is never taken for a rounding error,
# which is the whole of what it is for.
OXI_ROUNDING_TOL = 0.05        # e; a mean this far from an integer was not a
                               # state before it was rounded

# ── Looking a structure up ─────────────────────────────────────────────────
#
# v1.2.4: every lookup in this notebook was `df[df['source_cif'] == cif]`,
# which scans the whole frame. Measured on the perovskite run, 938 structures
# and 18,644 per-site rows:
#
#     one scan of the results frame                 2.4 ms
#     one groupby of the same frame                 0.11 s, once
#     one dict lookup afterwards                    0.0000013 ms
#
# About three scans per structure, and the classifier and the
# coordinate-screen count each did their own set, so batch_additivity took
# 5.10 s, classify_additivity 3.67 s and the counting loop 3.50 s.
#
# Group once, pass the index down. Every function that takes one takes it
# optionally and falls back to the scan, so it still works called on its own.
def index_by_cif(df, column='source_cif'):
    """Every row of df, grouped by structure, once."""
    return {k: v for k, v in df.groupby(column, sort=False)}


def rows_for(cif, df, index=None, column='source_cif'):
    """The rows for one structure, from the index if there is one.

    Returns an empty frame where the structure is absent, which is what the
    scan returned, so callers do not change.
    """
    if index is not None:
        got = index.get(cif)
        return got if got is not None else df.iloc[0:0]
    return df[df[column] == cif]


plt.rcParams.update({
    'font.family'      : 'sans-serif',
    'font.size'        : 11,
    'axes.labelsize'   : 13,
    'axes.titlesize'   : 14,
    'figure.dpi'       : 150,
    'savefig.dpi'      : 300,
    'savefig.bbox'     : 'tight',
    'axes.linewidth'   : 1.0,
    'xtick.direction'  : 'in',
    'ytick.direction'  : 'in',
    'xtick.major.width': 1.0,
    'ytick.major.width': 1.0,
})

print("Hoppe Additivity Analysis loaded.")

In [ ]:
# Uses a tkinter directory picker to select the folder containing
# maple_summary.csv and maple_results.csv from a MaplePy batch run.

import tkinter as tk
from tkinter import filedialog

# ── Where the MaplePy output is ───────────────────────────────────────────
#
# v1.2.5: running this notebook twice in one kernel could sit on this cell
# for minutes with nothing on screen. It is not the figures and it is not the
# CSVs: it is THIS DIALOG. A second Tk root in one process opens a dialog
# that, on Windows, frequently lands BEHIND the browser, and the cell then
# waits for a click on a window nobody can see. Restarting the kernel cures
# it, which is exactly what a hidden modal looks like.
#
# MaplePy's own picker passes parent=root and this one did not. It does now,
# with an update() either side so Tk actually maps the window, and the notice
# below is printed BEFORE the dialog blocks so a hang is diagnosable.
#
# Better still: set MAPLE_DIR_OVERRIDE and no dialog opens at all.
MAPLE_DIR_OVERRIDE = None      # e.g. r"path/to/outputs_YYYYMMDD_HHMMSS"

if MAPLE_DIR_OVERRIDE:
    MAPLE_DIR = str(MAPLE_DIR_OVERRIDE)
    print(f"Using MAPLE_DIR_OVERRIDE, no dialog: {MAPLE_DIR}")
else:
    print("Opening a folder picker. If nothing appears, look BEHIND this "
          "window - a second Tk dialog in one kernel often opens there. "
          "Set MAPLE_DIR_OVERRIDE above to skip the dialog entirely.")
    root = tk.Tk()
    root.withdraw()
    root.update()
    root.attributes('-topmost', True)
    root.lift()
    root.focus_force()

    MAPLE_DIR = filedialog.askdirectory(
        title="Select folder containing MaplePy output CSVs "
              "(maple_summary.csv and maple_results.csv)",
        parent=root
    )
    root.update()
    root.destroy()

if not MAPLE_DIR:
    raise FileNotFoundError(
        "No directory selected. Re-run this cell and choose the "
        "folder containing your MaplePy output CSVs."
    )

MAPLE_DIR = Path(MAPLE_DIR)
print(f"Selected directory: {MAPLE_DIR}")

# ── Locate CSVs ──────────────────────────────────────────────────────────
summary_csv = MAPLE_DIR / "maple_summary.csv"
results_csv = MAPLE_DIR / "maple_results.csv"

if not summary_csv.exists():
    # Check for outputs_* subdirectories (MaplePy timestamped folders)
    output_dirs = sorted(MAPLE_DIR.glob('outputs_*'), reverse=True)
    if output_dirs:
        MAPLE_DIR = output_dirs[0]
        summary_csv = MAPLE_DIR / 'maple_summary.csv'
        results_csv = MAPLE_DIR / 'maple_results.csv'
        print(f"  Using most recent output: {MAPLE_DIR.name}")

if not summary_csv.exists():
    raise FileNotFoundError(
        f"maple_summary.csv not found in {MAPLE_DIR}.\n"
        f"Run MaplePy batch processing (Section 5 or 7) first."
    )
if not results_csv.exists():
    raise FileNotFoundError(
        f"maple_results.csv not found in {MAPLE_DIR}.\n"
        f"Run MaplePy batch processing (Section 5 or 7) first."
    )

# ── Load ─────────────────────────────────────────────────────────────────
df_summary = pd.read_csv(summary_csv)
df_results = pd.read_csv(results_csv)

print(f"\nLoaded: {len(df_summary)} structures from {summary_csv.name}")
print(f"        {len(df_results)} site-level rows from {results_csv.name}")
print(f"        {df_summary['compound'].nunique()} unique compositions")
print(f"        {df_summary['space_group'].nunique()} space groups")
print(f"        {df_summary['disordered'].sum()} disordered structures")

## 1. Binary Oxide Reference MAPLE Table

See inline comments for full provenance documentation.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#
# Fallback values from MaplePy validation structures and Hoppe (1995).
# Format: {reduced_formula: (maple_kcal_mol, structure_type, source)}
#
# UPDATE these with your own MaplePy batch results on ICSD CIFs.
# The fallback values are a starting point, not a definitive reference.

BINARY_MAPLE_FALLBACK = {

    # ── Rock salt MO (Fm-3m) ──────────────────────────────────────────
    # ICSD median values
    "MgO":(1102.2, "Rock salt Fm-3m", "Median of 34 ambient ICSD Fm-3m entries, range 1092-1106"),
    "CaO":(965.0,  "Rock salt Fm-3m", "Median of 15 ambient ICSD Fm-3m entries, range 964-972"),
    "SrO":(900.0,  "Rock salt Fm-3m", "Median of 11 ambient ICSD Fm-3m entries, range 898-906"),
    "BaO":(838.5,  "Rock salt Fm-3m", "Median of 4 ambient ICSD Fm-3m entries, range 838-842; 1 zero excluded"),
    "MnO":(1044.7, "Rock salt Fm-3m", "Median of 13 ambient ICSD Fm-3m entries, range 1036-1053; 1 zero excluded"),
    "FeO":(1077.4, "Rock salt Fm-3m", "Median of 5 ambient ICSD Fm-3m entries, range 1073-1082; 2 zeros excluded"),
    "CoO":(1090.4, "Rock salt Fm-3m", "Median of 22 ambient ICSD Fm-3m entries, range 1086-1100"),
    "NiO":(1111.1, "Rock salt Fm-3m", "Median of 30 ambient ICSD Fm-3m entries, range 1052-1119"),
    "CdO":(988.6,  "Rock salt Fm-3m", "Median of 14 ambient ICSD Fm-3m entries, range 982-992; 3 zeros excluded"),
    "TiO":(1110.6, "Rock salt Fm-3m", "Median of 8 ambient ICSD Fm-3m entries, range 1081-1113; 2 zeros excluded"),
    "VO":(1136.8,  "Rock salt Fm-3m", "Median of 9 ambient ICSD Fm-3m entries, range 1127-1143; 1 zero excluded"),
    "ZnO":(1086.0, "Rock salt Fm-3m", "Median of 2 ambient ICSD Fm-3m entries, range 1085-1087; HP polymorph"),
    # Estimated rock salts
    "EuO":(902.9,  "Rock salt Fm-3m", "Median of 9 ambient ICSD Fm-3m entries, range 901-904; 3 zeros excluded"),
    "CuO":(1093.6, "Rock salt Fm-3m", "1 ambient ICSD Fm-3m entry; HP polymorph, not tenorite"),
    "PbO":(810.0,  "Litharge",         "Estimated, unsourced"),

    # ── Antifluorite M2O (Fm-3m) ──────────────────────────────────────
    # ICSD median values
    "Li2O":(837.7,  "Antifluorite Fm-3m", "Median of 10 ambient ICSD Fm-3m entries, range 835-838"),
    "Na2O":(694.5,  "Antifluorite Fm-3m", "Median of 2 ambient ICSD Fm-3m entries, range 694-695"),
    "K2O":(599.8,   "Antifluorite Fm-3m", "Median of 2 ambient ICSD Fm-3m entries, range 599-600"),
    "Rb2O":(572.6,  "Antifluorite Fm-3m", "Median of 2 ambient ICSD Fm-3m entries, range 572-573"),
    # Estimated
    "Cs2O":(450.0,  "Antifluorite Fm-3m", "Estimated, unsourced"),
    "Ag2O":(700.0,  "Cuprite Pn-3m",      "Estimated, unsourced"),
    "Cu2O":(730.0,  "Cuprite Pn-3m",      "MaplePy v4, a=4.270"),
    "Tl2O":(650.0,  "Various",            "Estimated, unsourced"),
    "Pd2O":(700.0,  "Various",            "Estimated, Pd1+ binary"),

    # ── Rutile MO2 (P4_2/mnm) ─────────────────────────────────────────
    # ICSD median values
    "TiO2":(3256.2, "Rutile P4_2/mnm", "Median of 112 ambient ICSD P4_2/mnm entries, range 3218-3505"),
    "SnO2":(3110.2, "Rutile P4_2/mnm", "Median of 48 ambient ICSD P4_2/mnm entries, range 3071-3156"),
    "MnO2":(3382.9, "Rutile P4_2/mnm", "Median of 19 ambient ICSD P4_2/mnm entries, range 3332-3388; 1 zero excluded"),
    "GeO2":(3389.6, "Rutile P4_2/mnm", "Median of 8 ambient ICSD P4_2/mnm entries, range 3386-3425"),
    "RuO2":(3244.8, "Rutile P4_2/mnm", "Median of 17 ambient ICSD P4_2/mnm entries, range 3228-3252"),
    "IrO2":(3225.5, "Rutile P4_2/mnm", "Median of 8 ambient ICSD P4_2/mnm entries, range 3221-3235"),
    "PbO2":(2955.0, "Rutile P4_2/mnm", "Median of 11 ambient ICSD P4_2/mnm entries, range 2946-2959"),
    "VO2":(3291.8,  "Rutile P4_2/mnm", "Median of 5 ambient ICSD P4_2/mnm entries, range 3284-3317"),
    "CrO2":(3352.8, "Rutile P4_2/mnm", "Median of 14 ambient ICSD P4_2/mnm entries, range 3346-3424"),
    "OsO2":(3210.8, "Rutile P4_2/mnm", "Median of 5 ambient ICSD P4_2/mnm entries, range 3205-3215"),
    "NbO2":(3153.4, "Rutile P4_2/mnm", "Median of 3 ambient ICSD P4_2/mnm entries, range 3120-3154"),
    "MoO2":(3179.7, "Rutile P4_2/mnm", "Median of 2 ambient ICSD P4_2/mnm entries, range 3179-3180"),
    "WO2":(3196.9,  "Rutile P4_2/mnm", "1 ambient ICSD P4_2/mnm entry"),
    "RhO2":(3264.4, "Rutile P4_2/mnm", "Median of 2 ambient ICSD P4_2/mnm entries, range 3261-3268; 1 zero excluded"),
    "PtO2":(3239.7, "Rutile P4_2/mnm", "1 ambient ICSD P4_2/mnm entry"),
    "PdO2":(3251.9, "Rutile P4_2/mnm", "1 ambient ICSD P4_2/mnm entry"),
    "ReO2":(3177.5, "Rutile P4_2/mnm", "1 ambient ICSD P4_2/mnm entry"),
    "TaO2":(3145.9, "Rutile P4_2/mnm", "Median of 2 ambient ICSD P4_2/mnm entries, range 3130-3162"),
    "SiO2":(3593.7, "Stishovite P4_2/mnm", "Median of 42 ambient ICSD P4_2/mnm entries, range 3575-3664"),
    # Estimated rutiles
    "TcO2":(3250.0, "Rutile P4_2/mnm", "Estimated from Ru/Rh/Os trend"),
    "SbO2":(3100.0, "Rutile P4_2/mnm", "Estimated, unsourced"),

    # ── Fluorite MO2 (Fm-3m) ──────────────────────────────────────────
    # ICSD median values
    "CeO2":(2856.3, "Fluorite Fm-3m", "Median of 64 ambient ICSD Fm-3m entries, range 2796-2891; 4 zeros excluded"),
    "ZrO2":(3020.0, "Fluorite Fm-3m", "Median of 12 ambient ICSD Fm-3m entries, range 3001-3176"),
    "HfO2":(3034.4, "Fluorite Fm-3m", "Median of 2 ambient ICSD Fm-3m entries, range 3016-3053"),
    "ThO2":(2761.6, "Fluorite Fm-3m", "Median of 15 ambient ICSD Fm-3m entries, range 2758-2770; 3 zeros excluded"),
    "UO2":(2826.6,  "Fluorite Fm-3m", "Median of 25 ambient ICSD Fm-3m entries, range 2821-2854; 2 zeros excluded"),
    "PrO2":(2865.5, "Fluorite Fm-3m", "Median of 5 ambient ICSD Fm-3m entries, range 2861-2872; 3 zeros excluded"),
    "TbO2":(2965.0, "Fluorite Fm-3m", "1 ambient ICSD Fm-3m entry; 2 zeros excluded"),
    "NpO2":(2843.9, "Fluorite Fm-3m", "Median of 2 ambient ICSD Fm-3m entries, range 2843-2844; 4 zeros excluded"),
    "AmO2":(2874.0, "Fluorite Fm-3m", "Median of 5 ambient ICSD Fm-3m entries, range 2867-2876"),
    "CmO2":(2879.4, "Fluorite Fm-3m", "Median of 2 ambient ICSD Fm-3m entries, range 2877-2882; 5 zeros excluded"),
    "PuO2":(2863.9, "Fluorite Fm-3m", "Median of 3 ambient ICSD Fm-3m entries, range 2863-2864; 2 zeros excluded"),

    # ── Corundum M2O3 (R-3c) ──────────────────────────────────────────
    # ICSD median values
    "Al2O3":(4356.2, "Corundum R-3c", "Median of 68 ambient ICSD R-3c entries, range 4250-4369"),
    "Cr2O3":(4170.3, "Corundum R-3c", "Median of 21 ambient ICSD R-3c entries, range 4159-4190"),
    "Fe2O3":(4121.9, "Corundum R-3c", "Median of 53 ambient ICSD R-3c entries, range 4074-4207; 2 outliers excluded"),
    "V2O3":(4124.1,  "Corundum R-3c", "Median of 14 ambient ICSD R-3c entries, range 4119-4924"),
    "Ti2O3":(4036.2, "Corundum R-3c", "Median of 12 ambient ICSD R-3c entries, range 4031-4066"),
    "Ga2O3":(4183.4, "Corundum R-3c", "Median of 3 ambient ICSD R-3c entries, range 4181-4198"),
    "Rh2O3":(4072.3, "Corundum R-3c", "Median of 4 ambient ICSD R-3c entries, range 4024-4087"),
    # Estimated corundums
    "Ni2O3":(4100.0, "Estimated",     "Interpolated from 3d corundum trend Ti-V-Cr-Fe; no stable ambient phase in ICSD"),

    # ── Spinel (Fd-3m) ────────────────────────────────────────────────
    # Co2O3 does not exist as a stable ambient phase; Co3O4 spinel used instead
    "Co3O4":(5445.2, "Spinel Fd-3m", "Median of 26 ambient ICSD Fd-3m entries, range 5020-5508"),

    # ── Bixbyite M2O3 (Ia-3) ─────────────────────────────────────────
    # ICSD median values
    "Mn2O3":(4133.9, "Bixbyite Ia-3", "Median of 10 ambient ICSD Ia-3 entries, range 4087-4155"),
    "Y2O3":(3652.0,  "Bixbyite Ia-3", "Median of 38 ambient ICSD Ia-3 entries, range 3585-4179; 2 zeros excluded"),
    "Sc2O3":(3929.6, "Bixbyite Ia-3", "Median of 8 ambient ICSD Ia-3 entries, range 3927-3966"),
    "Tl2O3":(3675.2, "Bixbyite Ia-3", "Median of 4 ambient ICSD Ia-3 entries, range 3663-3700"),
    "In2O3":(3822.4, "Bixbyite Ia-3", "Median of 18 ambient ICSD Ia-3 entries, range 3793-3849"),

    # ── RE sesquioxides ───────────────────────────────────────────────
    # ICSD median values — A-type (P-3m1) for La; bixbyite (Ia-3) for rest
    "La2O3":(3390.9, "A-type P-3m1",  "Median of 11 ambient ICSD P-3m1 entries, range 3337-3436"),
    "Ce2O3":(3483.6, "Bixbyite Ia-3", "1 ambient ICSD Ia-3 entry"),
    "Pr2O3":(3475.6, "Bixbyite Ia-3", "Median of 2 ambient ICSD Ia-3 entries, range 3476-3476"),
    "Nd2O3":(3496.8, "Bixbyite Ia-3", "Median of 3 ambient ICSD Ia-3 entries, range 3495-3507"),
    "Pm2O3":(3525.4, "Bixbyite Ia-3", "1 ambient ICSD Ia-3 entry"),
    "Sm2O3":(3545.1, "Bixbyite Ia-3", "Median of 5 ambient ICSD Ia-3 entries, range 3541-3558"),
    "Eu2O3":(3569.8, "Bixbyite Ia-3", "Median of 6 ambient ICSD Ia-3 entries, range 3559-3630"),
    "Gd2O3":(3581.4, "Bixbyite Ia-3", "Median of 14 ambient ICSD Ia-3 entries, range 3573-3611"),
    "Tb2O3":(3608.5, "Bixbyite Ia-3", "Median of 5 ambient ICSD Ia-3 entries, range 3592-3625"),
    "Dy2O3":(3631.2, "Bixbyite Ia-3", "Median of 10 ambient ICSD Ia-3 entries, range 3624-3646"),
    "Ho2O3":(3650.6, "Bixbyite Ia-3", "Median of 9 ambient ICSD Ia-3 entries, range 3641-3655"),
    "Er2O3":(3670.2, "Bixbyite Ia-3", "Median of 10 ambient ICSD Ia-3 entries, range 3655-3689"),
    "Tm2O3":(3698.6, "Bixbyite Ia-3", "Median of 5 ambient ICSD Ia-3 entries, range 3691-3710"),
    "Yb2O3":(3712.0, "Bixbyite Ia-3", "Median of 17 ambient ICSD Ia-3 entries, range 3707-3724"),
    "Lu2O3":(3727.0, "Bixbyite Ia-3", "Median of 12 ambient ICSD Ia-3 entries, range 3703-3735"),

    # ── Other binaries ────────────────────────────────────────────────
    "Bi2O3":(2580.0, "Monoclinic P2_1/c", "MaplePy v4, alpha-Bi2O3 — to be updated"),
    "HgO":(850.0,    "Various",            "Estimated, unsourced"),

    # ── Higher-valent oxides ──────────────────────────────────────────
    "V2O5":(9454.0,  "Pmmn",  "Median of 17 ambient ICSD Pmmn entries, range 9393-9638"),
    "Nb2O5":(8988.2, "C2/c",  "Median of 2 ambient ICSD C2/c entries, range 8984-8992"),
    "Ta2O5":(9127.2, "C2/c",  "1 ambient ICSD C2/c entry"),
    "Sb2O5":(8955.4, "C2/c",  "Median of 2 ambient ICSD C2/c entries, range 8955-8956"),
    "Bi2O5":(4300.0, "Various",     "Estimated, unsourced"),
    "WO3":(4600.0,   "ReO3-type",   "Estimated, unsourced"),
    "MoO3":(4500.0,  "Various",     "Estimated, unsourced"),
    "CrO3":(4400.0,  "Various",     "Estimated, unsourced"),
    "OsO3":(4500.0,  "Various",     "Estimated, unsourced"),
    "RuO3":(4500.0,  "Various",     "Estimated, Ru6+ binary"),
    "ReO3":(4600.0,  "ReO3-type",   "Estimated, Re6+ binary"),
    "Re2O5":(4400.0, "Various",     "Estimated, Re5+ binary"),
    "Re2O7":(5000.0, "Various",     "Estimated, unsourced"),
    "Tc2O7":(5000.0, "Various",     "Estimated, unsourced"),
    "Ru2O5":(4250.0, "Various",     "Estimated, Ru5+ binary"),
    "W2O5":(4300.0,  "Various",     "Estimated, W5+ binary"),
    "Mo2O3":(3300.0, "Various",     "Estimated, Mo3+ binary"),
}


def load_binary_table(csv_path=None, fallback=True):
    """
    Load the binary oxide MAPLE reference table.

    Priority:
    1. CSV from a MaplePy batch run on binary CIFs (if csv_path given)
    2. Fallback dict above

    Returns
    -------
    dict  {reduced_formula: maple_kcal_mol}
    pd.DataFrame  for display
    """
    lookup = {}
    rows = []

    if csv_path is not None and Path(csv_path).exists():
        df = pd.read_csv(csv_path)
        print(f"Loaded binary reference from {csv_path}")
        name_col = 'compound' if 'compound' in df.columns else 'formula'
        for _, r in df.iterrows():
            formula = r[name_col]
            maple = r['total_maple'] if 'total_maple' in df.columns \
                    else r.get('maple_kcal_mol')
            if pd.notna(maple):
                lookup[formula] = float(maple)
                rows.append({"formula": formula,
                             "maple_kcal_mol": float(maple),
                             "source": "ICSD CIF batch"})
        print(f"  {len(lookup)} binaries loaded from CSV.")

    if fallback:
        n_added = 0
        for formula, (maple, stype, source) in BINARY_MAPLE_FALLBACK.items():
            if formula not in lookup:
                lookup[formula] = maple
                rows.append({"formula": formula,
                             "maple_kcal_mol": maple,
                             "source": source})
                n_added += 1
        if n_added > 0:
            print(f"  {n_added} binaries added from fallback table.")

    df_out = pd.DataFrame(rows).sort_values("formula").reset_index(drop=True)
    print(f"  Total binary references: {len(lookup)}")
    return lookup, df_out


# ── The released reference table is the source ──────────────────────────────
#
# v1.2: BINARY_MAPLE_FALLBACK above is a 104-entry dict written inline. The
# released JSON has 112 - the extra eight are the oxyanion formers added for
# the borosulfate work, which the notebook never received. Measured formula by
# formula, no VALUE disagrees between the two, but the membership already
# does, and the copy the community downloads was not the copy the analysis
# used.
#
# The JSON is now read first. The inline dict remains for when the file is
# absent, and the guard below compares the two wherever they overlap: a value
# that has drifted stops the run and names the formula, rather than one copy
# quietly winning.

def load_binary_reference(path=None):
    """
    The binary oxide reference table, from the released JSON.

    Returns (lookup, source_string). Falls back to the inline dict only when
    no JSON can be found, and says so.
    """
    import json as _json
    from pathlib import Path as _Path

    _names = ('binary_oxide_maple_reference_v1_1.json',
              'binary_oxide_maple_reference.json')
    _dirs = ('data', '.', '..')
    _cands = ([_Path(path)] if path else
              [_Path(d) / n for d in _dirs for n in _names])

    for _p in _cands:
        if not _p.is_file():
            continue
        with open(_p, 'r', encoding='utf-8') as _f:
            _d = _json.load(_f)
        _lookup = {e['formula']: float(e['maple_kcal_mol'])
                   for e in _d['entries']}
        return _lookup, str(_p.resolve())

    if path:
        raise FileNotFoundError(f'binary reference table not found: {path}')
    return ({f: v[0] for f, v in BINARY_MAPLE_FALLBACK.items()},
            'INLINE FALLBACK - the released JSON was not found')


BINARY_REFERENCE_PATH = None    # set to override the search

_json_lookup, _binary_source = load_binary_reference(BINARY_REFERENCE_PATH)

# THE GUARD. Two copies of one table are safe only while something checks
# them. Where a formula is in both, the values must agree.
_drift = [(f, BINARY_MAPLE_FALLBACK[f][0], _json_lookup[f])
          for f in sorted(set(BINARY_MAPLE_FALLBACK) & set(_json_lookup))
          if abs(BINARY_MAPLE_FALLBACK[f][0] - _json_lookup[f]) > 1e-6]
if _drift:
    raise AssertionError(
        'the inline table and the released JSON disagree on '
        + ', '.join(f'{f} ({a} vs {b})' for f, a, b in _drift[:5])
        + f'{" and others" if len(_drift) > 5 else ""}. '
        'One of them is stale; decide which before running.')

print(f'  binary reference: {len(_json_lookup)} entries from {_binary_source}')

# ── Which entries are estimates ────────────────────────────────────────
#
# v1.2.9: 23 of the 112 values are estimates, interpolated within a structure
# type or set by hand, and none has a published or computed source; Cu2O's
# recorded source, a MaplePy calculation at a = 4.270 A, gives 797.9 rather
# than 730.0 and is marked unsourced in the JSON (sourced: false). They are
# kept, so that results made with them can be reproduced, and every
# decomposition that uses one says so (`estimated_binaries`). Nothing is
# reclassified on that account: a residual resting on an estimate is a
# residual against a weaker reference, and the reader is told which.
def estimated_binaries(source_path):
    """The formulae whose reference value is an estimate, or unsourced.

    From the released JSON's `source` and `sourced` fields where the table
    was read from it (Cu2O is unsourced, not estimated);
    from the inline labels where it was not.
    """
    import json as _json
    from pathlib import Path as _Path
    _p = _Path(source_path)
    if _p.is_file():
        with open(_p, 'r', encoding='utf-8') as _f:
            return {e['formula'] for e in _json.load(_f)['entries']
                    if e.get('source') == 'estimated'
                    or e.get('sourced') is False}
    return {f for f, (_v, _t, _s) in BINARY_MAPLE_FALLBACK.items()
            if _s.lower().startswith(('estimated', 'interpolated'))}


ESTIMATED_BINARIES = estimated_binaries(_binary_source)
print(f'  estimated or unsourced: {len(ESTIMATED_BINARIES)} entries')
print(f'  inline fallback:  {len(BINARY_MAPLE_FALLBACK)} entries, '
      f'{len(set(_json_lookup) - set(BINARY_MAPLE_FALLBACK))} of the released '
      f'entries not in it, no value in disagreement')

# To use your own ICSD batch results instead, set BINARY_CSV below.
BINARY_CSV = None   # e.g. Path("binary_oxides/maple_summary.csv")

BINARY_MAPLE, df_binary = load_binary_table(csv_path=BINARY_CSV)
# the released table wins over the inline fallback for anything it carries
BINARY_MAPLE.update(_json_lookup)
display(df_binary)

## 2. Ternary Decomposition Function

For an oxide A$_x$B$_y$O$_u$ with known cation oxidation states,
the decomposition into binary constituents is determined by
stoichiometry and charge balance.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────

def _binary_formula_for(element, oxi_state):
    """
    Return the expected binary oxide formula for a cation at a
    given oxidation state, plus the cation and oxygen counts
    per formula unit of that binary.

    Returns (formula_str, n_cations, n_oxygens) or (None, None, None).
    """
    oxi = round(abs(oxi_state))

    # ── Special cases ─────────────────────────────────────────────
    # Co3O4 spinel: Co2+[Co3+]2O4. Use as the Co3+ reference.
    # Decomposition: 1 Co3+ requires 1/3 Co3O4 (3 Co per fu),
    # contributing 4/3 O per Co3+.
    if element == "Co" and oxi == 3:
        return ("Co3O4", 3, 4)

    patterns = {
        1: (f"{element}2O",  2, 1),
        2: (f"{element}O",   1, 1),
        3: (f"{element}2O3", 2, 3),
        4: (f"{element}O2",  1, 2),
        5: (f"{element}2O5", 2, 5),
        6: (f"{element}O3",  1, 3),
        7: (f"{element}2O7", 2, 7),
    }
    return patterns.get(oxi, (None, None, None))


def decompose_ternary(source_cif, df_results, binary_lookup, verbose=True,
                      results_index=None):
    """
    Decompose a ternary oxide into binary components using the
    per-site oxidation states from MaplePy output.

    Parameters
    ----------
    source_cif     : str   filename from maple_summary/results
    df_results     : DataFrame  site-level MaplePy output
    binary_lookup  : dict  {formula: maple_kcal_mol}
    verbose        : bool

    Returns
    -------
    dict with keys:
        source_cif, compound, maple_ternary,
        decomposition (list of dicts per cation element),
        maple_binary_sum, delta_abs, delta_pct,
        oxygen_balanced, missing_binaries, success
    """
    # v1.2.4: from the index where the caller built one; the scan it
    # replaces cost 2.4 ms of the 5.4 ms this function took per structure.
    sites = rows_for(source_cif, df_results, results_index).copy()

    if sites.empty:
        return {"source_cif": source_cif, "success": False,
                "error": "No site data found"}

    compound     = sites['compound'].iloc[0]
    maple_tern   = sites['total_maple'].iloc[0]
    Z            = sites['Z'].iloc[0]

    # ── Identify cations and their mean oxidation states ──────────────
    cation_sites = sites[sites['oxidation_state'] > 0]
    anion_sites  = sites[sites['oxidation_state'] < 0]

    if cation_sites.empty:
        return {"source_cif": source_cif, "compound": compound,
                "success": False, "error": "No cation sites identified"}

    cation_info = {}
    for el in cation_sites['element'].unique():
        el_sites = cation_sites[cation_sites['element'] == el]
        cation_info[el] = {
            'count_per_fu': len(el_sites) / Z,
            'mean_oxi': el_sites['oxidation_state'].mean(),
            'sites_in_cell': len(el_sites),
        }

    o_per_fu = len(anion_sites) / Z

    # ── Decompose each cation into its binary ─────────────────────────
    decomposition = []
    maple_sum     = 0.0
    o_from_binary = 0.0
    missing       = []
    # v1.2.3: what was asked for and why it was not there, one entry per
    # missing binary. `missing_binaries` stays as it was, for the report and
    # for every CSV already written.
    missing_detail = []
    estimated_used = []     # v1.2.9: binaries whose value is an estimate

    for el, info in cation_info.items():
        binary_formula, n_cat, n_oxy = _binary_formula_for(
            el, info['mean_oxi']
        )

        # v1.2.3: did the pattern lookup have to round? A mean over a
        # shared site is not an oxidation state, and the rounding was silent.
        _mean = abs(info['mean_oxi'])
        _rounded = abs(_mean - round(_mean)) > OXI_ROUNDING_TOL

        if binary_formula is None:
            missing.append(f"{el}{info['mean_oxi']:+.1f} (no pattern)")
            missing_detail.append({
                'binary': None, 'element': el,
                'mean_oxi': round(info['mean_oxi'], 3),
                'rounded_from_non_integer': _rounded,
                'reason': 'no binary pattern for this oxidation state'})
            continue

        maple_binary = binary_lookup.get(binary_formula)
        if maple_binary is None:
            # v1.2.5: the rounding fact goes IN the string. It used to live in
            # missing_detail, a parallel list that batch_additivity never put
            # into its frame, so the classifier's best evidence never reached
            # it. One carrier, and it is the one that survives to the CSV.
            missing.append(
                f"{binary_formula} (not in table"
                + (f", mean {info['mean_oxi']:+.2f} rounded)" if _rounded
                   else ")"))
            missing_detail.append({
                'binary': binary_formula, 'element': el,
                'mean_oxi': round(info['mean_oxi'], 3),
                'rounded_from_non_integer': _rounded,
                'reason': 'not in the reference table'})
            continue

        # v1.2.9: an estimate is used, and said to be used
        if binary_formula in ESTIMATED_BINARIES:
            estimated_used.append(binary_formula)

        coeff = info['count_per_fu'] / n_cat
        contribution = coeff * maple_binary
        o_contrib    = coeff * n_oxy

        decomposition.append({
            'element'      : el,
            'oxi_state'    : round(info['mean_oxi'], 2),
            'binary'       : binary_formula,
            'n_cat_binary' : n_cat,
            'n_oxy_binary' : n_oxy,
            'coeff'        : round(coeff, 4),
            'maple_binary' : round(maple_binary, 2),
            'contribution' : round(contribution, 2),
            'o_from_binary': round(o_contrib, 4),
        })

        maple_sum     += contribution
        o_from_binary += o_contrib

    # ── Oxygen balance check ──────────────────────────────────────────
    #
    # v1.1: MEASURED, AND LEFT ALONE. This is an absolute tolerance on a
    # quantity that scales as 1/Z, which is the shape of a fault. Over 101
    # decompositions of real ICSD structures the imbalance is:
    #
    #        0 to 1e-6      97 structures      exact, to floating point
    #     1e-6 to 0.1        0
    #      0.1 to 0.2        3                 Co3O4 standing in for Co2O3
    #      0.2 and above     1                 Ca4Mn4O10.6
    #
    # Nothing lies inside the gap, so every value between about 1e-6 and 0.16
    # classifies identically and the number is not doing the work. What DID
    # matter was the Z error upstream: at Z = 1 the same physical imbalance
    # was multiplied by the true Z, close enough to the gap to cross it.
    # MaplePy v1.1 derives Z crystallographically and removes that.
    #
    # Note also that o_per_fu counts anion SITES, not occupancies, so an
    # oxygen-deficient refinement is balanced here and non-stoichiometric in
    # its composition. That is consistent with the mean-field treatment
    # elsewhere and is worth stating rather than discovering.
    O_BALANCE_TOL = 0.05          # O per formula unit; see the measurement above
    o_residual = abs(o_from_binary - o_per_fu)
    o_balanced = o_residual < O_BALANCE_TOL

    # ── Additivity residual ───────────────────────────────────────────
    #
    # v1.2.1: `if decomposition` asks whether ANY cation was decomposed, not
    # whether ALL were. 30 structures across the two families are missing a
    # binary reference, so their sum omitted a whole component and the
    # residual against it was up to 93% (NaOsO3: ternary 5022.2 against Na2O
    # alone at 347.2). success=False kept those out of the statistics, but the
    # number was still written to the CSV and Figure 1 selected on
    # delta.notna(), so all 30 sat in the histogram tail looking like
    # measured failures.
    #
    # Same rule as the falsy-zero fix above: do not write a number that has no
    # meaning. missing_binaries still says what was absent.
    complete = bool(decomposition) and not missing
    delta_abs = abs(maple_tern - maple_sum) if complete else None
    delta_pct = (100 * delta_abs / maple_tern
                 if delta_abs is not None and maple_tern > 0
                 else None)

    result = {
        'source_cif'       : source_cif,
        'compound'         : compound,
        'maple_ternary'    : round(maple_tern, 2),
        'Z'                : Z,
        'n_cation_elements': len(cation_info),
        'decomposition'    : decomposition,
        'maple_binary_sum' : round(maple_sum, 2),
        # v1.1: `if delta_abs` is FALSE for 0.0, so a structure whose
        # ternary MAPLE exactly equalled its binary sum - the perfect result
        # the theorem predicts - was written out as None and then read
        # downstream as a missing value. The guard in the classifier
        # attributed it to "a rounding or serialisation artefact"; it was
        # this line.
        'delta_abs'        : (round(delta_abs, 2)
                              if delta_abs is not None else None),
        'delta_pct'        : (round(delta_pct, 3)
                              if delta_pct is not None else None),
        'oxygen_per_fu'    : round(o_per_fu, 4),
        'oxygen_from_bins' : round(o_from_binary, 4),
        'oxygen_balanced'  : o_balanced,
        'missing_binaries' : missing,
        'missing_detail'   : missing_detail,
        'estimated_binaries': estimated_used,
        'success'          : len(decomposition) > 0 and len(missing) == 0,
    }

    if verbose and result['success']:
        status = (f"GOOD (<{ADDITIVE_THRESHOLD_PCT:g}%)"
                  if delta_pct < ADDITIVE_THRESHOLD_PCT else
                  "MARGINAL (1-2%)" if delta_pct < 2.0 else
                  "POOR (>2%)")
        print(f"  {compound:20s}  "
              f"MAPLE={maple_tern:8.1f}  "
              f"sum={maple_sum:8.1f}  "
              f"delta={delta_pct:.2f}%  {status}"
              f"{'  [O imbalanced]' if not o_balanced else ''}")
    elif verbose and not result['success']:
        print(f"  {compound:20s}  INCOMPLETE: missing {missing}")

    return result


print("Decomposition function defined.")

## 3. Batch Additivity Calculation

Run the decomposition across all structures in the MaplePy output
and assemble results into a summary DataFrame.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────

def batch_additivity(df_summary, df_results, binary_lookup,
                     verbose=False):
    """
    Compute additivity residuals for all structures.

    Parameters
    ----------
    df_summary    : DataFrame from maple_summary.csv
    df_results    : DataFrame from maple_results.csv
    binary_lookup : dict {formula: maple_kcal_mol}
    verbose       : bool

    Returns
    -------
    pd.DataFrame  one row per structure with additivity results
    list of dicts  full decomposition details (for inspection)
    """
    rows = []
    details = []

    # v1.2.4: once, here, rather than a scan per structure inside the loop.
    results_index = index_by_cif(df_results)

    for _, summary_row in df_summary.iterrows():
        cif = summary_row['source_cif']

        result = decompose_ternary(
            cif, df_results, binary_lookup, verbose=verbose,
            results_index=results_index
        )
        details.append(result)

        rows.append({
            'source_cif'       : cif,
            'compound'         : result.get('compound', ''),
            'space_group'      : summary_row['space_group'],
            'Z'                : result.get('Z'),
            'disordered'       : summary_row['disordered'],
            'maple_ternary'    : result.get('maple_ternary'),
            'maple_binary_sum' : result.get('maple_binary_sum'),
            'delta_abs'        : result.get('delta_abs'),
            'delta_pct'        : result.get('delta_pct'),
            'madelung_factor'  : summary_row['madelung_factor'],
            'd_ref_ang'        : summary_row['d_ref_ang'],
            'oxygen_balanced'  : result.get('oxygen_balanced'),
            'n_cation_elements': result.get('n_cation_elements'),
            'missing_binaries' : '; '.join(result.get('missing_binaries', [])),
            'estimated_binaries': '; '.join(result.get('estimated_binaries', [])),
            'success'          : result.get('success', False),
        })

    df = pd.DataFrame(rows)

    # Summary statistics
    n_total   = len(df)
    n_success = df['success'].sum()
    n_fail    = n_total - n_success

    if n_success > 0:
        good     = (df['success'] & (df['delta_pct'] < 1.0)).sum()
        marginal = (df['success'] & (df['delta_pct'] >= 1.0)
                    & (df['delta_pct'] < 2.0)).sum()
        poor     = (df['success'] & (df['delta_pct'] >= 2.0)).sum()
    else:
        good = marginal = poor = 0

    print(f"\n{'='*65}")
    print(f"  Additivity analysis: {n_total} structures")
    print(f"  Decomposed successfully: {n_success}")
    print(f"    Good (<1%):     {good}")
    print(f"    Marginal (1-2%): {marginal}")
    print(f"    Poor (>2%):     {poor}")
    print(f"  Incomplete (missing binaries): {n_fail}")
    _n_est = (df['success'] & (df['estimated_binaries'] != '')).sum()
    print(f"  Resting on an estimated binary: {_n_est} of the {n_success} "
          f"decomposed (see estimated_binaries)")
    print(f"{'='*65}")

    return df, details


# ── Run the batch ────────────────────────────────────────────────────────────
print("Computing additivity residuals for all structures...")
df_additivity, additivity_details = batch_additivity(
    df_summary, df_results, BINARY_MAPLE, verbose=False
)

# Show the successful decompositions
df_ok = df_additivity[df_additivity['success']].sort_values('delta_pct')
print(f"\nSuccessfully decomposed: {len(df_ok)} structures")
if len(df_ok) > 0:
    display(df_ok[['compound','space_group','Z','maple_ternary',
                    'maple_binary_sum','delta_abs','delta_pct',
                    'oxygen_balanced','disordered']].head(30))

## 4. Failure Classification — Tiered Numeric Scheme

| Code | Tier | Label |
|------|------|-------|
| Additive | 0 | Additive (Δ < 1%) |
| 1.1 | 1 | Reference table incomplete |
| 1.2 | 1 | Upstream calculation failure (always zero here) |
| 1.3 | 1 | No binary end-member exists |
| 1.4 | 1 | Oxidation state has no binary analogue |
| 2.1 | 2 | Oxidation state assignment error |
| 2.2 | 2 | Z = 1 metadata artefact (cannot arise from MaplePy) |
| 2.3 | 2 | Disorder / mean-field |
| 2.4 | 2 | Oxygen non-stoichiometry |
| 2.5 | 2 | Structural coordinate error |
| 2.6 | 2 | Anion vacancy disorder |
| 2.7 | 2 | Split-site disorder model |
| 3.1 | 3 | Genuine violation |

**Order:** Tier 1 (1.1, 1.3, 1.4); then the structural defects, 2.7 and 2.5, whatever the residual; then Additive; then 2.1 → 2.6 → 2.3 → 2.4; then 3.1. A caveat must not mask a defect. The registry in the cell below defines every code.

In [ ]:
# 4. Failure Classification — Tiered Numeric Scheme
# ─────────────────────────────────────────────────────────────────────────────
#
# Every structure receives a classification:
#
#   Additive         Δ < 1%, no failure code assigned
#   Tier 1 (1.x)     Calculation incomplete
#   Tier 2 (2.x)     Calculation complete, result compromised
#   Tier 3 (3.x)     Calculation complete, result trustworthy
#
# v1.2.9: the order as classify_additivity() applies it (this header had
# kept the first version's): Tier 1 (1.1, 1.3, 1.4); the structural
# defects, 2.7 then 2.5, whatever the residual; Additive; then 2.1, 2.6,
# 2.3, 2.4; then 3.1. A caveat must not mask a defect. 1.2 and 2.2 are
# defined and cannot arise from a MaplePy run (see the registry).

# ── Classification registry ──────────────────────────────────────────────
# Central definition of all codes.  Add new codes here; the classifier
# and all downstream plotting/summary code reads from this dict.

CLASSIFICATION_REGISTRY = {
    # code : (tier, label, description)
    # v1.2.3: narrowed. 1.1 now means only what its name says: the binary
    # oxide exists and OUR TABLE lacks it, which is a job of work rather than
    # a result. The other reasons a binary can be absent have their own codes.
    '1.1': (1, 'Reference table incomplete',
            'The binary oxide exists at ambient conditions and is absent from '
            'the reference table. Adding it makes the structure testable.'),
    # v1.2.6: defined properly rather than 'reserved'. These structures are
    # real and they are counted, just not here: MaplePy writes them to
    # maple_failed.csv and they never produce a row for Hopper to classify.
    # Three perovskite CIFs in this study, out of 941 retrieved.
    '1.2': (1, 'Upstream calculation failure',
            'MaplePy could not compute a MAPLE value from the CIF, so there '
            'is no ternary value to test. Refined occupancies exceeding '
            'unity and parse failures both land here. These entries appear '
            'in maple_failed.csv and are excluded before classification, so '
            'this code is always zero in a Hopper output.'),
    # v1.2.3: the most interesting exclusion in the study, and it used to be
    # reported as a missing table entry. Fe(IV), Co(IV), Ru(III), Cu(III),
    # Pd(III), Os(V), Tc(V), Rh(V) and Ir(V) are real states in these
    # perovskites and pyrochlores and none of them has an ambient binary
    # oxide. The theorem cannot be tested against an end-member that does not
    # exist.
    '1.3': (1, 'No binary end-member exists',
            'The oxidation state is real and the corresponding binary oxide '
            'does not exist at ambient conditions, so the decomposition has '
            'no end-member to sum. A limit on where additivity can be tested, '
            'not a defect in the structure or the table.'),
    # v1.2.5: a THIRD route, found once the rounding evidence could
    # actually reach the classifier. Nd4Cu4O11.724 asks for Cu2O3 from a mean
    # of +2.86 and Nd4Ru3.544O12 asks for Ru2O3 from +3.38: means over a
    # non-stoichiometric refinement, not over a shared site. Same conclusion,
    # third way of getting there, and it is in the description now.
    '1.4': (1, 'Oxidation state has no binary analogue',
            'The charge the decomposition asked for is not a real oxidation '
            'state of that element: a formal average over a '
            'charge-disproportionated site, an occupancy-weighted mean over a '
            'site shared by two cations, or a mean over a non-stoichiometric '
            'refinement. Where the mean was not an integer the decomposition '
            'records it and this code is assigned on that evidence.'),
    '2.1': (2, 'Oxidation state assignment error',
            'Delta > 10% in an ordered, oxygen-balanced structure. Points to '
            'incorrect oxidation states in the CIF.'),
    # v1.2.6: RETAINED, not retired. MaplePy v1.1 stopped inferring Z from
    # composition and derives it from the gcd of the site multiplicities, so
    # this cannot fire on a MaplePy result and is zero throughout this study.
    # It is kept because the failure mode is not MaplePy's: any tool that
    # reads Z from the CIF, or trusts a compositional reduction, will produce
    # it, and anyone adopting this scheme needs somewhere to put it. Zero
    # here is a result about the calculator, not an empty slot.
    '2.2': (2, 'Z = 1 metadata artefact',
            'The number of formula units per cell has been taken as 1 when '
            'the cell contains more, inflating MAPLE per formula unit by an '
            'integer factor and raising the Madelung factor well above the '
            'family range. Arises wherever Z is inferred from composition '
            'rather than derived from the site multiplicities.'),
    '2.3': (2, 'Disorder / mean-field',
            'Mixed-occupancy sites processed via mean-field charge '
            'approximation. Residual may reflect averaging artefact.'),
    '2.4': (2, 'Oxygen non-stoichiometry',
            'Oxygen count from binary decomposition does not match ternary. '
            'Decomposition not well-defined (e.g. Co3O4 for Co2O3).'),
    # v1.1: the count has been removed from this description. It read 32
    # here, 19 in the detector's docstring and 31 in the manuscript, all
    # shipping together. A number that lives in a string drifts from the data
    # it counts; the notebook prints the population it actually found.
    # v1.2.2: two detectors now, either of which is sufficient. The CV test
    # needs a 20-site Pnma cell with 12 oxygens; the distance test needs
    # nothing but coordinates. Each catches cases the other cannot: 22 of the
    # 26 perovskites under 1.6 A were already caught by CV, and 8 structures
    # caught by CV have an ordinary 1.98 A shortest contact.
    '2.5': (2, 'Structural coordinate error',
            'Either the O-site MAPLE coefficient of variation exceeds 0.05, '
            'or the shortest interatomic contact is below MaplePy\'s '
            'SHORT_CONTACT_ANG. Both indicate unphysical interatomic '
            'distances from Wyckoff site mis-assignment or wrong fractional '
            'coordinates.'),
    # v1.2.2: ACTIVE. Reserved since the first version and assignable now
    # that MaplePy v1.3.3 reports which sublattice carries the fractional
    # occupancy. Tested before 2.3, because an oxygen-deficient refinement
    # described as "disorder" has lost the part that matters.
    '2.6': (2, 'Anion vacancy disorder',
            'Fractional occupancy on the anion sublattice only: an '
            'oxygen-deficient refinement, treated by mean-field charge '
            'averaging. Distinct from cation disorder (2.3) and from a '
            'decomposition that cannot balance its oxygen (2.4).'),
    '2.7': (2, 'Split-site disorder model',
            'Two sites closer than MaplePy\'s SPLIT_SITE_ANG: one '
            'crystallographic position split by a disorder model. The Ewald '
            'sum over fractionally occupied near-coincident sites is not an '
            'energy, whatever the residual comes out as.'),
    '3.1': (3, 'Genuine violation',
            'Delta >= 1% with correct oxidation states, correct Z, balanced '
            'oxygen, no disorder. Reflects genuine structural chemistry.'),
}

# ── The codes, in the order they are reported ──────────────────────────────
# v1.2.1: this order was written out three times (twice here, once in the
# figure cell), and the figure's copy omitted 1.1 and 2.3, so those rows were
# in the count and not in the bars.
# v1.2.6: derived, not typed. The hand-written list named the codes that
# actually fire, which meant 1.2 and 2.2 were absent from every breakdown
# rather than present and zero. A code defined in the scheme and reported
# nowhere is indistinguishable from a code nobody thought of.
CODE_ORDER = ['Additive'] + sorted(CLASSIFICATION_REGISTRY)


# ── Why a binary oxide is not in the table ─────────────────────────────────
#
# v1.2.3: one entry per binary the decompositions have actually asked for and
# not found, with the reason beside it. This is chemistry, so it is written
# down and auditable rather than buried in a branch, and anything NOT here
# classifies 1.1 and says it is unclassified - a new missing binary announces
# itself instead of joining a pile.
#
#   'table'      the binary exists at ambient conditions; we lack it
#   'no_binary'  the oxidation state is real; no ambient binary oxide of it
#   'no_state'   the charge is not a real oxidation state of that element
#
MISSING_BINARY_STATUS = {
    # the binary exists and the table should have it
    'SnO':   ('table',
              'Litharge-type SnO is the ambient Sn(II) oxide. Ta2Sn2O7 has '
              'Sn(II) with Ta(V), which balances, so this is a genuine gap.'),
    'PtO':   ('table',
              'PtO is reported as a tetragonal ambient phase, though Pt3O4 '
              'and PtO2 are better characterised. Needs a verified structure '
              'before a MAPLE value is added.'),
    'SmO':   ('table',
              'Rock-salt SmO is a real Sm(II) phase. NOTE: the structure that '
              'asks for it, SmTiO3 ICSD 51180, is assigned Sm2+ with Ti4+, '
              'while the known compound is Sm3+Ti3+O3. Both balance to 6, so '
              'charge neutrality cannot see the difference. The oxidation '
              'state assignment deserves a look.'),

    # the state is real; the binary oxide is not
    'Ru2O3': ('no_binary', 'Ru(III) has no ambient binary oxide; RuO2 is the '
                           'stable oxide.'),
    'FeO2':  ('no_binary', 'Fe(IV) has no ambient binary oxide. CaFeO3 and '
                           'EuFeO3 really do contain Fe(IV).'),
    'CoO2':  ('no_binary', 'Co(IV) has no ambient binary oxide; CoO2 exists '
                           'only as a delithiated layered phase.'),
    'Cu2O3': ('no_binary', 'Cu(III) has no ambient binary oxide; Cu2O and '
                           'CuO are the binaries.'),
    'Pd2O3': ('no_binary', 'Pd(III) has no ambient binary oxide.'),
    'Os2O5': ('no_binary', 'Os(V) has no ambient binary oxide; OsO2 and OsO4 '
                           'are the known ones.'),
    'Tc2O5': ('no_binary', 'Tc(V) has no ambient binary oxide; TcO2 and '
                           'Tc2O7 are the known ones.'),
    'Rh2O5': ('no_binary', 'Rh(V) has no ambient binary oxide; Rh2O3 and '
                           'RhO2 are the binaries.'),
    'Ir2O5': ('no_binary', 'Ir(V) has no ambient binary oxide; IrO2 is the '
                           'stable oxide.'),

    # the charge is not a state
    'BiO2':  ('no_state',
              'Bi(IV) is a formal average. SrBiO3 is charge-disproportionated '
              'Bi(III)/Bi(V), and the CIF deposits Bi as 4+ on every site, so '
              'the rounding test cannot catch it.'),
    'YO2':   ('no_state', 'Y(IV) does not exist. Y3NbO7 shares its cation '
                          'site with Nb and the site mean is 3.5.'),
    'EuO2':  ('no_state', 'Eu(IV) does not exist. Eu2Sn2O7 shares its cation '
                          'site with Sn and the site mean is 3.5.'),
    'TlO':   ('no_state', 'Tl(II) does not exist; Tl is 1+ or 3+. Nb2Tl2O7 '
                          'shares its cation site with Nb.'),
}

_MISSING_STATUS_CODE = {'table': '1.1', 'no_binary': '1.3', 'no_state': '1.4'}


def as_missing_list(value):
    """
    The missing-binary entries as a list, whatever the caller has.

    v1.2.5: batch_additivity stores `'; '.join(missing)`, so the classifier
    was handed a STRING and iterated it character by character - every one of
    the 20 tier-1 perovskites came out 1.4 with a reason built from single
    letters. Every test I wrote passed a list, because I built the frames
    myself or read them back with literal_eval. One place converts now, and
    it takes a list, a joined string, or the repr of a list.
    """
    if value is None:
        return []
    if isinstance(value, (list, tuple)):
        return [v for v in value if v]
    if not isinstance(value, str):
        return []                       # NaN from an empty CSV cell
    text = value.strip()
    if not text or text in ('[]', 'nan'):
        return []
    if text.startswith('[') and text.endswith(']'):
        import ast as _ast
        try:
            lit = _ast.literal_eval(text)
            if isinstance(lit, (list, tuple)):
                return [v for v in lit if v]
        except (ValueError, SyntaxError):
            pass
    return [p.strip() for p in text.split(';') if p.strip()]


def _rounded_from_missing(entry):
    """Whether this request came from a rounded site mean, per the string."""
    return isinstance(entry, str) and 'rounded)' in entry


def _binary_from_missing(entry):
    """The binary formula out of one `missing_binaries` string, or None.

    The strings are written by decompose_ternary as 'Ru2O3 (not in table)'
    and 'Fe+4.0 (no pattern)'. Only the first form names a binary.
    """
    if not isinstance(entry, str):
        return None
    head = entry.split(' (')[0].strip()
    return head if '(not in table)' in entry else None


def classify_missing_binaries(missing, detail=None):
    # v1.2.5: `detail` is kept in the signature for anything that still
    # passes it, and ignored: the rounding fact is in the string now.
    """
    Which code a decomposition that lacked a binary belongs in, and why.

    v1.2.3: 1.1 held four different findings. Returns (code, reason).

    The rounding evidence wins where it exists: a request built on a
    non-integral site mean is 1.4 whatever the registry says, because the
    charge was never an oxidation state. Otherwise the registry decides, and
    an unregistered binary stays 1.1 and says so.

    Where a structure is missing more than one binary the strongest finding
    wins, in the order 1.4, 1.3, 1.1: a structure whose decomposition rests
    on a charge that does not exist is not waiting on our table.
    """
    reasons, codes = [], []
    for entry in as_missing_list(missing):
        formula = _binary_from_missing(entry)

        if _rounded_from_missing(entry):
            codes.append('1.4')
            reasons.append(f'{entry}: asked for from a rounded site mean, '
                           f'which was not an oxidation state')
            continue

        if formula is None:
            codes.append('1.4')
            reasons.append(f'{entry}: no binary pattern for that charge')
            continue

        status = MISSING_BINARY_STATUS.get(formula)
        if status is None:
            codes.append('1.1')
            reasons.append(f'{formula}: UNCLASSIFIED, not in '
                           f'MISSING_BINARY_STATUS')
            continue
        codes.append(_MISSING_STATUS_CODE[status[0]])
        reasons.append(f'{formula}: {status[1]}')

    if not codes:
        return '1.1', 'decomposition failed with no binary named'
    for code in ('1.4', '1.3', '1.1'):
        if code in codes:
            return code, '; '.join(reasons)
    return '1.1', '; '.join(reasons)


# ── The population a residual statistic is drawn from ──────────────────────
def additivity_population(df):
    """
    The structures whose residual IS a residual: decomposition complete and
    delta computed.

    v1.2.1: Figure 1 selected on delta.notna() while the classifier counted
    tier 0, so the figure said 662 within 1% and the classification said 661
    additive, on the same run and in the same paper. One definition, used by
    both.
    """
    return df[df['success'].astype(bool) & df['delta_pct'].notna()]


def n_classified_additive(df):
    """How many structures the classifier called additive. One number."""
    return int((additivity_population(df)['classification'] == 'Additive').sum())


# Convenience lookups
def _tier_for_code(code):
    """Return the tier for a classification outcome.

    v1.2.7: 'Additive' is an outcome without a registry code, and it is
    tier 0. Callers that map over CODE_ORDER meet it first, so answering
    here is better than every caller special-casing it.
    """
    if code == 'Additive':
        return 0
    return CLASSIFICATION_REGISTRY[code][0]

def _label_for_code(code):
    """Return the human-readable label for a classification code."""
    return CLASSIFICATION_REGISTRY[code][1]

"""
Code 2.5 detector: Wyckoff 4a/4b site mis-assignment in Pnma perovskites.

Detects ICSD entries where the B-site cation is deposited at Wyckoff 4a
instead of the correct 4b in GdFeO3-type perovskites (Pnma, 20 sites).

v1.1: the counts that used to sit here have been removed. They disagreed
with the registry and with the manuscript, and the notebook reports the
population it finds.

v1.2.9: the usage example that stood here named a function that does not
exist. The detector is detect_structural_coordinate_error(), below, and
classify_additivity() calls it; since v1.2.2, code 2.5 also takes a
short contact from MaplePy's geometry_flag.
"""


def detect_structural_coordinate_error(cif_name, results_df, summary_df,
                                        cv_threshold=0.05,
                                        results_index=None,
                                        summary_index=None):
    """
    Detect structural coordinate errors in Pnma perovskites using the
    coefficient of variation (CV) of per-site oxygen MAPLE values.
 
    In a correctly refined GdFeO3-type perovskite (Pnma, Z=4, 20 sites),
    the 12 oxygen sites occupy two crystallographically distinct positions
    (Wyckoff 4c and 8d) but experience similar electrostatic environments,
    giving per-site MAPLE values within ~5% of each other (CV < 0.045).
 
    Structural coordinate errors distort the local coordination geometry
    and produce anomalously different MAPLE values between O1 and O2
    sublattices, pushing CV above 0.05.
 
    Parameters
    ----------
    cif_name : str
        Source CIF filename (must match a row in summary_df).
    results_df : pd.DataFrame
        Per-site MAPLE results with columns: source_cif, element,
        maple_kcal_mol.
    summary_df : pd.DataFrame
        Per-structure summary with columns: source_cif, Z, space_group.
    cv_threshold : float
        CV threshold above which the structure is flagged.
        Default 0.05 (validated gap: 0.045-0.050).
 
    Returns
    -------
    bool or None
        True  — structural coordinate error detected (classify as 2.5).
        False — structure is clean.
        None  — structure is not a testable Pnma perovskite.
    """
    srow = rows_for(cif_name, summary_df, summary_index)
    if len(srow) == 0:
        return None
    srow = srow.iloc[0]
 
    # v1.1: THE TEST NO LONGER GATES ON Z.
    #
    # This asked for Z == 4, so every perovskite whose Z had been lost to
    # composition reduction - 101 of the 938, all non-stoichiometric - was
    # returned as untestable and never screened for coordinate errors at all.
    # The claim of zero false positives across 938 was really across 831.
    #
    # What the test actually needs is a 20-site Pnma cell with 12 oxygens,
    # which is checked below and does not require Z. With MaplePy v1.1
    # deriving Z from the site counts the two conditions coincide anyway, and
    # a test should not depend on a quantity it does not use.
    if srow['space_group'] != 'Pnma':
        return None
 
    entry = rows_for(cif_name, results_df, results_index)
    if len(entry) != 20:
        return None
 
    oxygens = entry[entry['element'] == 'O']
    if len(oxygens) != 12:
        return None
 
    # Coefficient of variation of per-site oxygen MAPLE
    o_maples = oxygens['maple_kcal_mol'].values.astype(float)
    o_mean = np.mean(o_maples)
 
    if o_mean <= 0:
        return None
 
    cv = float(np.std(o_maples) / o_mean)
 
    # Explicit bool() to avoid numpy.bool_ identity issues
    return bool(cv > cv_threshold)

def _summary_field(cif_name, summary_df, column, summary_index=None):
    """
    One field of MaplePy's summary for one structure, or None.

    v1.2.2: the three tests below all need the same lookup, and all three
    must answer None rather than False when MaplePy did not write the
    column - a test that cannot run is not a test that passed.
    """
    if column not in summary_df.columns:
        return None
    srow = rows_for(cif_name, summary_df, summary_index)
    if len(srow) == 0:
        return None
    value = srow[column].iloc[0]
    return value if isinstance(value, str) else None


def detect_split_site(cif_name, summary_df, summary_index=None):
    """
    Whether MaplePy flagged this structure as a split-site disorder model.

    v1.2.2: reads `geometry_flag`, written by MaplePy v1.3.3. It used to read
    the prefix of `geometry_status`, which is a sentence written for a human:
    Hopper's classification should not depend on MaplePy's wording.

    Hopper does not recompute the geometry. MaplePy holds the structures and
    the tolerances, and two implementations of one test is how they come to
    disagree.
    """
    flag = _summary_field(cif_name, summary_df, 'geometry_flag',
                           summary_index)
    return None if flag is None else flag == 'split_site'


def detect_short_contact(cif_name, summary_df, summary_index=None):
    """
    Whether the shortest contact is below MaplePy's SHORT_CONTACT_ANG.

    v1.2.2: the second detector for code 2.5. The CV test needs a 20-site
    Pnma cell with 12 oxygens and cannot see a short contact anywhere else;
    NdFeO3 at 1.507 A was classified Additive at delta 0.312% because of it.
    """
    flag = _summary_field(cif_name, summary_df, 'geometry_flag',
                           summary_index)
    return None if flag is None else flag == 'short_contact'


def detect_anion_vacancy(cif_name, summary_df, summary_index=None):
    """
    Whether the fractional occupancy is on the anion sublattice alone.

    v1.2.2: code 2.6, reserved since the first version and assignable now
    that MaplePy v1.3.3 writes `partial_sublattice`. Anion-only, not 'both':
    a structure with cation disorder as well is a mean-field result for more
    than one reason and belongs in 2.3.
    """
    where = _summary_field(cif_name, summary_df, 'partial_sublattice',
                           summary_index)
    return None if where is None else where == 'anion'


def classify_additivity(df_add, results_df, summary_df,
                        threshold_additive=ADDITIVE_THRESHOLD_PCT):
    """
    Assign a tiered numeric classification to each structure based on
    its additivity residual and metadata.

    Parameters
    ----------
    df_add : DataFrame
        Output from batch_additivity(), one row per structure.
    results_df, summary_df : DataFrame
        MaplePy's per-site and per-structure output. v1.1: these were read
        as free variables from the notebook namespace, so re-running the
        cells out of order classified against a stale frame silently.
    threshold_additive : float
        Residual threshold (%) below which a structure is classified
        as additive.  Default 1.0 (Hoppe criterion).

    Returns
    -------
    DataFrame with added columns:
        classification       str   'Additive' or code ('1.1', '2.1', etc.)
        classification_tier  int   0 for additive, 1/2/3 for failures
        classification_label str   human-readable label

    Notes
    -----
    v1.2.9: the order is the one in the header of this cell, not
    ascending code number: Tier 1; then 2.7 and 2.5 whatever the
    residual; then Additive; then 2.1, 2.6, 2.3, 2.4; then 3.1. The
    first that applies is assigned.
    """
    df = df_add.copy()
    # v1.2.4: two indexes, built once, for every lookup below.
    _ridx = index_by_cif(results_df) if len(results_df) else None
    _sidx = index_by_cif(summary_df) if len(summary_df) else None
    df['classification']       = ''
    df['classification_tier']  = 0
    df['classification_label'] = ''
    df['hopper_version']       = HOPPER_VERSION
    df['missing_binary_reason'] = ''

    # v1.2.1: say when a test cannot run. Against CSVs written before MaplePy
    # v1.3.2 there is no geometry column, and a split-site structure would
    # then be classified on whatever else it trips - which is how all four of
    # them came to be called oxygen non-stoichiometry.
    for _col, _tests in (('geometry_flag', 'the split-site test (2.7) and '
                          'the short-contact half of 2.5'),
                         ('partial_sublattice', 'the anion-vacancy test (2.6)')):
        if _col not in summary_df.columns:
            print(f"  NOTE: no {_col} column in the summary. These CSVs were "
                  f"written by MaplePy before v1.3.3, so {_tests} did not "
                  f"run.")

    for idx, row in df.iterrows():

        # ── Codes 1.1, 1.3, 1.4: the decomposition lacked a binary ──
        # v1.2.3: three findings, not one. Which of them is decided by
        # classify_missing_binaries, from the rounding evidence where
        # there is any and from the registry where there is not.
        if not row['success'] and as_missing_list(
                row['missing_binaries']):
            _code, _why = classify_missing_binaries(row['missing_binaries'])
            df.at[idx, 'classification']       = _code
            df.at[idx, 'classification_tier']   = 1
            df.at[idx, 'classification_label']  = (
                f'{_label_for_code(_code)} ({_why})' if _code == '1.1'
                and 'UNCLASSIFIED' in _why else _label_for_code(_code))
            df.at[idx, 'missing_binary_reason'] = _why
            if 'UNCLASSIFIED' in _why:
                print(f"  NOTE: {row['source_cif']} asks for a binary that is "
                      f"not in MISSING_BINARY_STATUS - {_why}")
            continue

        # Catch-all for other decomposition failures (should not occur
        # with current data, but defensive)
        if not row['success']:
            df.at[idx, 'classification']       = '1.1'
            df.at[idx, 'classification_tier']   = 1
            df.at[idx, 'classification_label']  = 'Decomposition failed'
            continue

        # ── Guard: success=True but delta_pct unavailable ───────────
        # Can occur when maple_ternary == maple_binary_sum exactly,
        # producing delta_abs = 0.0 that becomes NaN through a
        # rounding or serialisation artefact.  Treat as additive.
        if row['delta_pct'] is None or (isinstance(row['delta_pct'], float)
                                         and row['delta_pct'] != row['delta_pct']):
            df.at[idx, 'classification']       = 'Additive'
            df.at[idx, 'classification_tier']   = 0
            df.at[idx, 'classification_label']  = 'Additive'
            continue

        # ── Code 2.7: split-site disorder model ────────────────────
        # Checked before Additive, like 2.5: a structure whose two
        # half-occupied sites are 0.03 A apart is not additive when its
        # residual lands under 1%, it is unmeasured. MaplePy v1.3.2
        # flags these; see its SPLIT_SITE_ANG for the tolerance.
        if detect_split_site(row['source_cif'], summary_df, _sidx):
            df.at[idx, 'classification']       = '2.7'
            df.at[idx, 'classification_tier']   = 2
            df.at[idx, 'classification_label']  = _label_for_code('2.7')
            continue

        # ── Code 2.5: structural coordinate error ──────────────────
        # Two detectors, either sufficient, and the label says which
        # found it. The CV test sees a B-site at 4a instead of 4b in a
        # 20-site Pnma cell; the distance test sees any contact below
        # SHORT_CONTACT_ANG anywhere.
        #
        # Both are checked before Additive, because DyCrO3 138222 passes
        # the 1% threshold by coincidence despite the Wyckoff error and
        # NdFeO3 passes it with two atoms 1.507 A apart.
        _cv = detect_structural_coordinate_error(
            row['source_cif'], results_df, summary_df,
            results_index=_ridx, summary_index=_sidx)
        _short = detect_short_contact(row['source_cif'], summary_df, _sidx)
        if _cv or _short:
            _how = ('O-site MAPLE CV and short contact' if _cv and _short
                    else 'short contact' if _short else 'O-site MAPLE CV')
            df.at[idx, 'classification']       = '2.5'
            df.at[idx, 'classification_tier']   = 2
            df.at[idx, 'classification_label']  = (
                f"{_label_for_code('2.5')} ({_how})")
            continue
        
        # ── Additive: delta < threshold ─────────────────────────────
        # Check additive status before entering the failure logic.
        # Structures within Hoppe range get no failure code.
        if (row['delta_pct'] is not None
                and row['delta_pct'] < threshold_additive):
            df.at[idx, 'classification']       = 'Additive'
            df.at[idx, 'classification_tier']   = 0
            df.at[idx, 'classification_label']  = 'Additive'
            continue

        # ── From here, delta >= threshold: assign failure codes ─────
        # Applied in ascending code order so the lowest applicable
        # code wins.

        # ── Code 2.1: oxidation state assignment error ──────────────
        # Very large residual (>10%) in an ordered structure with
        # balanced oxygen points to incorrect oxidation states in the
        # CIF, producing a wrong binary decomposition.
        if (row['delta_pct'] is not None and row['delta_pct'] > 10.0
                and not row['disordered'] and row['oxygen_balanced']):
            df.at[idx, 'classification']       = '2.1'
            df.at[idx, 'classification_tier']   = 2
            df.at[idx, 'classification_label']  = _label_for_code('2.1')
            continue

        # ── Code 2.2: defined, and unreachable from a MaplePy result ──
        # There was a branch here assigning 2.2 to every Z = 1 structure,
        # written when composition-derived Z made Z = 1 a reliable sign of
        # an artefact. MaplePy v1.1 derives Z from the site counts, the
        # population fell 107 -> 2, and the two survivors are cubic
        # perovskites whose Z really is 1. So no branch assigns 2.2 here:
        # a Z = 1 structure from MaplePy is classified on its residual like
        # any other. The CODE stays in the registry, because the artefact is
        # real for any tool that reads Z from the CIF or trusts a
        # compositional reduction, and it is reported as zero rather than
        # omitted.

        # ── Code 2.6: anion vacancy disorder ───────────────────────
        # Fractional occupancy on the anion sublattice alone: an
        # oxygen-deficient refinement. Tested before 2.3 because it is
        # the more specific description of the same structure, and
        # before 2.4 because it says WHY the oxygen does not balance.
        if detect_anion_vacancy(row['source_cif'], summary_df, _sidx):
            df.at[idx, 'classification']       = '2.6'
            df.at[idx, 'classification_tier']   = 2
            df.at[idx, 'classification_label']  = _label_for_code('2.6')
            continue

        # ── Code 2.3: disorder / mean-field approximation ───────────
        # Structure has mixed-occupancy sites processed via mean-field
        # charge averaging.  The residual may partly reflect the
        # averaging artefact rather than genuine structural chemistry.
        if row['disordered']:
            df.at[idx, 'classification']       = '2.3'
            df.at[idx, 'classification_tier']   = 2
            df.at[idx, 'classification_label']  = _label_for_code('2.3')
            continue

        # ── Code 2.4: oxygen non-stoichiometry ──────────────────────
        # Oxygen count from binary decomposition does not match the
        # ternary oxygen count.  Principal cause: Co3O4 used in place
        # of the non-existent Co2O3.
        if not row['oxygen_balanced']:
            df.at[idx, 'classification']       = '2.4'
            df.at[idx, 'classification_tier']   = 2
            df.at[idx, 'classification_label']  = _label_for_code('2.4')
            continue

        # ── Code 3.1: genuine violation ─────────────────────────────
        # delta >= threshold with correct oxidation states, correct Z,
        # balanced oxygen, no disorder.  The residual reflects genuine
        # structural chemistry (covalency, strain, etc.).
        if row['delta_pct'] is not None and row['delta_pct'] >= threshold_additive:
            df.at[idx, 'classification']       = '3.1'
            df.at[idx, 'classification_tier']   = 3
            df.at[idx, 'classification_label']  = _label_for_code('3.1')

    return df


# ── Classify ─────────────────────────────────────────────────────────────────
df_classified = classify_additivity(df_additivity, df_results, df_summary)

# v1.1: the 2.5 population, counted from the data rather than from a string.
_n25 = int((df_classified['classification'] == '2.5').sum())
# v1.2.4: the same question as before - how many structures the screen
# COULD test, which is not the same as how many the classifier reached,
# since a structure classified 1.3 or 2.7 never gets to the 2.5 branch.
# So the loop stays, and the indexes make it cost 0.1 s instead of 3.5.
_ridx_screen = index_by_cif(df_results)
_sidx_screen = index_by_cif(df_summary)
_tested = sum(1 for _c in df_classified['source_cif']
              if detect_structural_coordinate_error(
                  _c, df_results, df_summary,
                  results_index=_ridx_screen,
                  summary_index=_sidx_screen) is not None)
print(f'  coordinate-error screen: {_n25} flagged of {_tested} testable '
      f'({len(df_classified)} structures in the run)')

# ── Summary by tier ──────────────────────────────────────────────────────────
print("Classification summary")
print("=" * 65)

# Tier-level summary
tier_labels = {0: 'Additive', 1: 'Tier 1 (incomplete)',
               2: 'Tier 2 (compromised)', 3: 'Tier 3 (trustworthy)'}
for tier_num in [0, 1, 2, 3]:
    count = (df_classified['classification_tier'] == tier_num).sum()
    if count > 0:
        print(f"  {tier_labels[tier_num]:30s}  {count:5d}")
print()

# Code-level summary
print("Breakdown by classification code:")
for code in CODE_ORDER:
    subset = df_classified[df_classified['classification'] == code]
    if len(subset) > 0:
        label = subset['classification_label'].iloc[0]
        print(f"  {code:10s}  {label:40s}  {len(subset):5d}")
print("=" * 65)
_pop = additivity_population(df_classified)
print(f"  residual population: {len(_pop)} of {len(df_classified)} "
      f"(decomposition complete), of which "
      f"{n_classified_additive(df_classified)} classified additive at "
      f"delta < {ADDITIVE_THRESHOLD_PCT:g}%")
print()

# ── Show each classification group ──────────────────────────────────────────
for code in CODE_ORDER:
    subset = df_classified[df_classified['classification'] == code]
    if len(subset) > 0:
        label = subset['classification_label'].iloc[0]
        tier  = subset['classification_tier'].iloc[0]
        tier_str = f"Tier {tier}" if tier > 0 else "Additive"
        print(f"\n{code} — {label} [{tier_str}] ({len(subset)} structures)")
        cols = ['compound', 'space_group', 'Z', 'delta_pct',
                'disordered', 'oxygen_balanced']
        if code == '1.1':
            cols.append('missing_binaries')
        display(subset[cols].head(15))


## 5. Additivity Residual Distribution (Key Paper Figure)

Histogram of Δ% for successfully decomposed structures,
with Hoppe’s ±1% range marked.  Coloured by classification code.

In [ ]:
# 5a. Additivity Residual Histogram (Key Paper Figure)
# ─────────────────────────────────────────────────────────────────────────────
#
# Two-panel figure:
#   (a) 0-5% at 0.25% bins — shows the bulk of the data
#   (b) >5% at 2.5% bins   — shows the failure tail in detail
#
# Colour scheme (Sheffield brand palette):
#   Additive      pastel_green   (light, recessive — the expected result)
#   3.1 Genuine   teal           (stands out against the green)
#   2.4 O-nonstoich flamingo     (warm pink)
#   2.2 Z=1       peach          (warm orange)
#   2.1 Oxi state coral          (warm red)

from matplotlib.lines import Line2D

# ── Sheffield-palette colour map for classification codes ────────────────
CLASSIFICATION_COLOURS = {
    'Additive': SHEFFIELD['pastel_green'],
    '1.1':      SHEFFIELD['pale_violet'],
    '1.2':      SHEFFIELD['aqua'],
    '1.3':      SHEFFIELD['peak_green'],
    '1.4':      SHEFFIELD['spearmint'],
    '2.1':      SHEFFIELD['coral'],
    '2.2':      SHEFFIELD['peach'],
    '2.3':      SHEFFIELD['powder_violet'],
    '2.4':      SHEFFIELD['flamingo'],
    '2.5':      SHEFFIELD['brand_violet'],
    '2.6':      SHEFFIELD['powder_blue'],
    '2.7':      SHEFFIELD['midnight'],
    '3.1':      SHEFFIELD['teal'],
}

CLASSIFICATION_LABELS = {
    'Additive': rf'Additive ($\Delta$ < {ADDITIVE_THRESHOLD_PCT:g}%)',
    '1.1':      '1.1 Table incomplete',
    '1.2':      '1.2 Upstream failure',
    '1.3':      '1.3 No binary end-member',
    '1.4':      '1.4 Charge is not a state',
    '2.1':      '2.1 Oxi state error',
    '2.2':      '2.2 Z = 1 artefact (retired)',
    '2.3':      '2.3 Disorder / mean-field',
    '2.4':      '2.4 O non-stoichiometry',
    '2.5':      '2.5 Coordinate error',
    '2.6':      '2.6 Anion vacancy',
    '2.7':      '2.7 Split-site model',
    '3.1':      '3.1 Genuine violation',
}

# Marker shapes per tier for greyscale accessibility
TIER_MARKERS = {0: 'o', 1: 'v', 2: 's', 3: 'D'}


def plot_additivity_histogram(df_classified, save_path=None):
    """
    Two-panel histogram of additivity residuals.

    Panel (a): 0-5% range at 0.25% bins with Hoppe 1% threshold.
    Panel (b): >5% tail at 2.5% bins showing failure breakdown.

    Coloured by classification code using the Sheffield brand palette.
    Lighter pastel_green for additive structures so the failure codes
    (warm colours + teal) stand out.
    """
    # v1.2.1: the same population the classifier reports, from the same
    # function. This selected on delta.notna() alone, which included the 30
    # structures whose binary sum was incomplete (their delta is None now)
    # and produced a "within 1%" count one larger than the classifier's
    # additive count, on the same run.
    df_plot = additivity_population(df_classified).copy()

    if len(df_plot) == 0:
        print("No structures with computed residuals to plot.")
        return

    fig, (ax_main, ax_tail) = plt.subplots(
        1, 2, figsize=(12, 5),
        gridspec_kw={'width_ratios': [3, 2], 'wspace': 0.25}
    )

    # Stacking order: additive at the back, then by code.
    #
    # v1.2.1: the old order here omitted 1.1 and 2.3, so those structures
    # were counted in the annotation and drawn in neither panel. Every code
    # present is drawn now, and a code with no colour raises rather than
    # disappearing.
    code_order = ['Additive'] + [c for c in CODE_ORDER if c != 'Additive'][::-1]
    codes_present = [c for c in code_order
                     if c in df_plot['classification'].values]
    _uncoloured = [c for c in df_plot['classification'].unique()
                   if c not in CLASSIFICATION_COLOURS]
    if _uncoloured:
        raise KeyError(f'no colour for classification code(s) {_uncoloured}: '
                       f'add them to CLASSIFICATION_COLOURS and '
                       f'CLASSIFICATION_LABELS rather than plotting without')
    colours = [CLASSIFICATION_COLOURS[c] for c in codes_present]
    labels = [CLASSIFICATION_LABELS[c] for c in codes_present]

    # ── Panel (a): 0 to 5% ──────────────────────────────────────────
    bins_main = np.arange(0, 5.25, 0.25)
    df_main = df_plot[df_plot['delta_pct'] <= 5.0]

    data_main = [df_main[df_main['classification'] == c]['delta_pct'].values
                 for c in codes_present]

    ax_main.hist(data_main, bins=bins_main, color=colours, label=labels,
                 edgecolor='white', linewidth=0.5, stacked=True)

    # Hoppe's threshold, from the constant the classifier uses
    ax_main.axvline(x=ADDITIVE_THRESHOLD_PCT, color=SHEFFIELD['coral'],
                    linestyle='--', linewidth=1.5, alpha=0.8, zorder=5)
    ax_main.text(1.12, ax_main.get_ylim()[1] * 0.95,
                 f"Hoppe {ADDITIVE_THRESHOLD_PCT:g}%",
                 fontsize=10, color=SHEFFIELD['coral'], va='top')

    ax_main.set_xlabel(r'Additivity residual, $\Delta$ / %')
    ax_main.set_ylabel('Count')
    ax_main.set_title('(a)', loc='left', fontweight='bold')
    ax_main.spines['top'].set_visible(False)
    ax_main.spines['right'].set_visible(False)
    ax_main.set_xlim(-0.1, 5.2)
    ax_main.legend(frameon=False, fontsize=9, loc='upper right')

    # Annotation: the classifier's own additive count, not a second count
    # made here against the same threshold over a different population.
    n_within = n_classified_additive(df_classified)
    n_total = len(df_plot)
    ax_main.text(0.97, 0.12,
                 f'{n_within}/{n_total} ({100*n_within/n_total:.0f}%)\n'
                 'classified additive',
                 transform=ax_main.transAxes, ha='right', va='bottom',
                 fontsize=10, color=SHEFFIELD['dark_violet'], style='italic')

    # ── Panel (b): > 5%, uniform 2.5% bins ──────────────────────────
    df_tail = df_plot[df_plot['delta_pct'] > 5.0]
    bins_tail = np.arange(5, 100, 2.5)

    codes_tail = [c for c in code_order
                  if c in df_tail['classification'].values]
    data_tail = [df_tail[df_tail['classification'] == c]['delta_pct'].values
                 for c in codes_tail]
    colours_tail = [CLASSIFICATION_COLOURS[c] for c in codes_tail]

    ax_tail.hist(data_tail, bins=bins_tail, color=colours_tail,
                 edgecolor='white', linewidth=0.5, stacked=True)

    ax_tail.set_xlabel(r'Additivity residual, $\Delta$ / %')
    ax_tail.set_ylabel('Count')
    ax_tail.set_title(r'(b) $\Delta$ > 5%' + f'  (n = {len(df_tail)})',
                       loc='left', fontweight='bold')
    ax_tail.spines['top'].set_visible(False)
    ax_tail.spines['right'].set_visible(False)
    ax_tail.set_xlim(5, 100)

    fig.tight_layout()

    if save_path:
        fig.savefig(save_path)
        print(f"Saved: {save_path}")

    plt.show()
    return fig


# ── Plot ─────────────────────────────────────────────────────────────────────
fig_hist = plot_additivity_histogram(
    df_classified,
    save_path=MAPLE_DIR / 'fig1_additivity_histogram.png'
)

In [ ]:
# 5b. Madelung Factor Clustering (Key Paper Figure)
# ─────────────────────────────────────────────────────────────────────────────
#
# Two-panel figure:
#   (a) All structures — overview with labelled common phases
#   (b) Z=1 cluster detail — auto-detected zoom window with ICSD codes
#
# Panel (b) window is auto-detected from the gap in the MF distribution
# between the main perovskite band and the Z=1 cluster.  Override by
# setting the PANEL_B_* variables below to explicit values.
#
# Label strategy:
#   Panel (a): common phases (n >= 5) at median positions
#   Panel (b): clustered compositions (n >= 3) with entry counts,
#              edge singletons/pairs with compound + ICSD CollCode
#
# Marker shapes per tier for greyscale accessibility:
#   Additive (Tier 0): circles
#   Tier 2 (compromised): squares
#   Tier 3 (genuine): diamonds

from adjustText import adjust_text
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

# ── User-configurable panel (b) window ───────────────────────────────────
# Set to explicit values to override auto-detection.
# Set to None to auto-detect from the data.
PANEL_B_MF_MIN = None       # e.g. 8.0
PANEL_B_MF_MAX = None       # e.g. 17.5
PANEL_B_DREF_MIN = None     # e.g. 1.6
PANEL_B_DREF_MAX = None     # e.g. 2.3

# ── Compounds to label in panel (a) ─────────────────────────────────────
# Edit this list to control which phases are labelled in the main band.
# Only compounds with MF < 6 (i.e. in the main perovskite band) are
# labelled; the list is ignored for compounds not in the data.
LABEL_COMPOUNDS_PANEL_A = [
    'MgSiO3', 'CaMnO3', 'CaTiO3', 'LaMnO3', 'LaFeO3',
    'NdNiO3', 'BaCeO3', 'SrCeO3', 'NaNbO3', 'NaTaO3',
    'SrRuO3', 'LaCrO3',
]

# ── Tier-to-marker mapping ──────────────────────────────────────────────
# v1.2.7: was a hand-written map missing 1.3, 1.4, 2.6 and 2.7, so those
# structures were dropped from every scatter. _tier_for_code has been in
# cell 12 the whole time and reads the registry.
tier_map = {c: _tier_for_code(c) for c in CODE_ORDER}


def _plot_scatter_by_tier(ax, data, size_map, alpha_additive=0.5):
    """Plot scatter points with tier-based marker shapes and classification colours."""
    for tier in [0, 2, 3]:
        subset = data[data['tier'] == tier]
        if len(subset) == 0:
            continue
        if tier == 2:
            # v1.2.7: every tier-2 code in the registry, not four of seven
            for code in [c for c in CODE_ORDER
                         if _tier_for_code(c) == 2]:
                sub = subset[subset['classification'] == code]
                if len(sub) == 0:
                    continue
                ax.scatter(sub['d_ref_ang'], sub['madelung_factor'],
                          c=CLASSIFICATION_COLOURS[code],
                          marker=TIER_MARKERS[tier], s=size_map.get(tier, 22),
                          alpha=0.75, edgecolors='white', linewidth=0.3,
                          zorder=3)
        else:
            code = 'Additive' if tier == 0 else '3.1'
            ax.scatter(subset['d_ref_ang'], subset['madelung_factor'],
                      c=CLASSIFICATION_COLOURS[code],
                      marker=TIER_MARKERS[tier], s=size_map.get(tier, 22),
                      alpha=alpha_additive if tier == 0 else 0.8,
                      edgecolors='white', linewidth=0.3,
                      zorder=2 if tier == 0 else 4)


def plot_mf_clustering(df_classified, save_path=None):
    """
    Two-panel Madelung factor vs d_ref scatter plot.

    Panel (a): all successfully decomposed structures, with common
    phases labelled at their median position.  Dashed box indicates
    the region shown in panel (b).

    Panel (b): zoomed view of the Z=1 / high-MF cluster.  Edge and
    outlier ICSD entries are labelled with compound + CollCode;
    clustered compositions (n >= 3) are labelled with entry count.

    The panel (b) window is auto-detected from the largest gap in the
    MF distribution above the main perovskite band, unless overridden
    by the PANEL_B_* variables.
    """
    ok = df_classified[df_classified['success'] == True].copy()
    ok['tier'] = ok['classification'].map(tier_map).fillna(-1).astype(int)
    ok['coll_code'] = ok['source_cif'].str.extract(r'CollCode(\d+)', expand=False)

    # ── Auto-detect panel (b) window ────────────────────────────────
    # Find the first significant gap (> 2.0) in MF above the main band
    mf_sorted = ok['madelung_factor'].sort_values().values
    all_gaps = np.diff(mf_sorted)
    mf_gap_lower = 5.0
    mf_gap_upper = 10.0
    for i in range(len(all_gaps)):
        if all_gaps[i] > 2.0 and mf_sorted[i] > 4.0:
            mf_gap_lower = mf_sorted[i]
            mf_gap_upper = mf_sorted[i + 1]
            break

    auto_mf_min = mf_gap_upper - 0.5

    # Exclude extreme outliers (e.g. SrRuO3 at MF~34)
    high_cluster = ok[(ok['madelung_factor'] >= auto_mf_min) &
                       (ok['madelung_factor'] < 20)]

    if len(high_cluster) > 0:
        auto_mf_max = high_cluster['madelung_factor'].max() + 0.5
        auto_dref_min = high_cluster['d_ref_ang'].min() - 0.05
        auto_dref_max = high_cluster['d_ref_ang'].max() + 0.05
    else:
        auto_mf_max = 18.0
        auto_dref_min = 1.6
        auto_dref_max = 2.3

    pb_mf_min = PANEL_B_MF_MIN if PANEL_B_MF_MIN is not None else auto_mf_min
    pb_mf_max = PANEL_B_MF_MAX if PANEL_B_MF_MAX is not None else auto_mf_max
    pb_dref_min = PANEL_B_DREF_MIN if PANEL_B_DREF_MIN is not None else auto_dref_min
    pb_dref_max = PANEL_B_DREF_MAX if PANEL_B_DREF_MAX is not None else auto_dref_max

    print(f"Panel (b) window: MF [{pb_mf_min:.1f}, {pb_mf_max:.1f}], "
          f"d_ref [{pb_dref_min:.2f}, {pb_dref_max:.2f}]")

    # ── Build figure ────────────────────────────────────────────────
    # v1.2.7: panel (b) zoomed the Z = 1 cluster. With Z derived from site
    # multiplicities that cluster is empty, and the panel was drawing an
    # empty axis titled 'n = 0' next to a dashed box around nothing. Decide
    # from the data: no cluster, no panel.
    cluster_b = ok[(ok['madelung_factor'] >= pb_mf_min) &
                   (ok['madelung_factor'] <= pb_mf_max) &
                   (ok['d_ref_ang'] >= pb_dref_min) &
                   (ok['d_ref_ang'] <= pb_dref_max)].copy()

    if len(cluster_b) == 0:
        print('No high-MF cluster: panel (b) omitted.')
        fig, ax_a = plt.subplots(1, 1, figsize=(9, 6.5))
        ax_b = None
    else:
        fig, (ax_a, ax_b) = plt.subplots(
            1, 2, figsize=(14, 6.5),
            gridspec_kw={'width_ratios': [3, 2], 'wspace': 0.28}
        )

    # ── Panel (a): all structures ───────────────────────────────────
    _plot_scatter_by_tier(ax_a, ok, {0: 18, 2: 22, 3: 22})

    # Labels for common phases in the main band
    grouped_main = ok[ok['madelung_factor'] < 6].groupby('compound').agg(
        n=('compound', 'size'),
        mf_med=('madelung_factor', 'median'),
        dref_med=('d_ref_ang', 'median'),
    ).reset_index()

    texts_a = []
    for comp in LABEL_COMPOUNDS_PANEL_A:
        row = grouped_main[grouped_main['compound'] == comp]
        if len(row) == 0:
            continue
        r = row.iloc[0]
        n_str = f' ({int(r["n"])})' if r['n'] >= 5 else ''
        texts_a.append(ax_a.text(r['dref_med'], r['mf_med'],
                                 f'{comp}{n_str}',
                                 fontsize=7, color=SHEFFIELD['midnight'],
                                 fontstyle='italic'))

    # Label the SrRuO3 outlier at MF~34
    sr_out = ok[(ok['compound'] == 'SrRuO3') & (ok['madelung_factor'] > 30)]
    if len(sr_out) > 0:
        r = sr_out.iloc[0]
        texts_a.append(ax_a.text(r['d_ref_ang'], r['madelung_factor'],
                                 f'SrRuO3\nCollCode{r["coll_code"]}',
                                 fontsize=7, color=SHEFFIELD['midnight'],
                                 fontstyle='italic', ha='right'))

    adjust_text(texts_a, ax=ax_a,
                arrowprops=dict(arrowstyle='-', color='grey', alpha=0.4, lw=0.5),
                expand=(1.5, 1.5), force_text=(0.8, 0.8),
                ensure_inside_axes=True)

    # Gap reference line
    gap_y = (mf_gap_lower + mf_gap_upper) / 2
    ax_a.axhline(y=gap_y, color='grey', linestyle=':', linewidth=0.8, alpha=0.4)

    # Box showing panel (b) region, only if there is a panel (b)
    if ax_b is not None:
     rect = Rectangle((pb_dref_min, pb_mf_min),
                      pb_dref_max - pb_dref_min,
                      pb_mf_max - pb_mf_min,
                      linewidth=1.0, edgecolor=SHEFFIELD['dark_violet'],
                      facecolor='none', linestyle='--', alpha=0.6, zorder=5)
     ax_a.add_patch(rect)
     ax_a.text(pb_dref_max + 0.01, pb_mf_max, '(b)',
               fontsize=9, color=SHEFFIELD['dark_violet'], va='top',
               fontweight='bold')

    ax_a.set_xlabel('$d_\\mathrm{ref}$ / Å')
    ax_a.set_ylabel('Madelung factor')
    if ax_b is not None:
        ax_a.set_title('(a)', loc='left', fontweight='bold')
    ax_a.spines['top'].set_visible(False)
    ax_a.spines['right'].set_visible(False)

    # v1.2.7: built from the registry, and only for codes that actually
    # appear in this dataset. The hand-written version listed 2.2, which is
    # now always empty, and omitted 1.3, 1.4, 2.3, 2.6 and 2.7, which are not.
    _present = [c for c in CODE_ORDER if (ok['classification'] == c).any()]
    legend_elements = [
        Line2D([0], [0],
               marker={0: 'o', 3: 'D'}.get(_tier_for_code(c), 's'),
               color='w', markerfacecolor=CLASSIFICATION_COLOURS[c],
               markeredgecolor='white', markersize=7 if c != '3.1' else 6,
               label=CLASSIFICATION_LABELS[c])
        for c in _present
    ]
    ax_a.legend(handles=legend_elements, frameon=False, fontsize=8,
                loc='upper left', handletextpad=0.5)

    # ── Panel (b): zoomed cluster, when there is one ────────────────
    if ax_b is None:
        fig.tight_layout()
        if save_path:
            fig.savefig(save_path)
            print(f'  saved {save_path}')
        return fig

    print(f"Panel (b): {len(cluster_b)} structures in window")
    _plot_scatter_by_tier(ax_b, cluster_b, {0: 35, 2: 40, 3: 40},
                          alpha_additive=0.6)

    # ── Label strategy for panel (b) ────────────────────────────────
    cluster_grouped = cluster_b.groupby('compound').agg(
        n=('compound', 'size'),
        mf_med=('madelung_factor', 'median'),
        dref_med=('d_ref_ang', 'median'),
        cls=('classification', 'first'),
        codes=('coll_code', lambda x: list(x)),
    ).reset_index()

    texts_b = []

    # Clustered compositions (n >= 3): label at median with count
    clustered = cluster_grouped[cluster_grouped['n'] >= 3]
    for _, r in clustered.iterrows():
        texts_b.append(ax_b.text(
            r['dref_med'], r['mf_med'],
            f'{r["compound"]} ({int(r["n"])})',
            fontsize=6.5, color=SHEFFIELD['midnight'],
            fontstyle='italic', fontweight='bold'))

    # Edge/outlier points: singletons and pairs at the edges
    if len(cluster_b) > 0:
        mf_hi = cluster_b['madelung_factor'].quantile(0.92)
        mf_lo = cluster_b['madelung_factor'].quantile(0.08)
        dref_hi = cluster_b['d_ref_ang'].quantile(0.92)
        dref_lo = cluster_b['d_ref_ang'].quantile(0.08)

        edge_candidates = cluster_grouped[cluster_grouped['n'] <= 2]
        for _, r in edge_candidates.iterrows():
            is_edge = (r['mf_med'] >= mf_hi or r['mf_med'] <= mf_lo or
                       r['dref_med'] >= dref_hi or r['dref_med'] <= dref_lo)
            if is_edge:
                cc = r['codes'][0] if r['codes'] else ''
                n_str = f' ({int(r["n"])})' if r['n'] == 2 else ''
                label = f'{r["compound"]}{n_str}\n{cc}' if cc else r['compound']
                texts_b.append(ax_b.text(
                    r['dref_med'], r['mf_med'], label,
                    fontsize=5.5, color=SHEFFIELD['midnight'],
                    fontstyle='italic'))

    if texts_b:
        adjust_text(texts_b, ax=ax_b,
                    arrowprops=dict(arrowstyle='-', color='grey',
                                    alpha=0.4, lw=0.5),
                    expand=(2.0, 2.0), force_text=(1.2, 1.2),
                    ensure_inside_axes=True)

    ax_b.set_xlabel('$d_\\mathrm{ref}$ / Å')
    ax_b.set_ylabel('Madelung factor')
    ax_b.set_title(f'(b) Z = 1 cluster detail  (n = {len(cluster_b)})',
                    loc='left', fontweight='bold')
    ax_b.spines['top'].set_visible(False)
    ax_b.spines['right'].set_visible(False)
    ax_b.set_xlim(pb_dref_min, pb_dref_max)
    ax_b.set_ylim(pb_mf_min, pb_mf_max)

    fig.subplots_adjust(wspace=0.28)

    if save_path:
        fig.savefig(save_path)
        print(f"Saved: {save_path}")

    plt.show()
    return fig


# ── Plot ─────────────────────────────────────────────────────────────────────
fig_mf = plot_mf_clustering(
    df_classified,
    save_path=MAPLE_DIR / 'fig2_mf_clustering.png'
)

## 6. Perovskite-Specific Analysis

Filter for perovskite space groups and examine MF clustering.

In [ ]:
# 6. Perovskite-Specific Analysis
# ─────────────────────────────────────────────────────────────────────────────

PEROVSKITE_SGS = ['Pnma', 'Pm-3m', 'R-3c', 'P4mm', 'I4/mcm',
                  'Cmcm', 'R3c', 'P2_1/n', 'Imma']

df_perov = df_classified[
    df_classified['space_group'].isin(PEROVSKITE_SGS)
].copy()

print(f"Perovskite candidates: {len(df_perov)} structures")
print(f"  Space groups: {df_perov['space_group'].value_counts().to_dict()}")
print(f"  Successfully decomposed: {df_perov['success'].sum()}")

# Classification breakdown for perovskites
print(f"\n  Classification breakdown:")
for code in CODE_ORDER:
    n = (df_perov['classification'] == code).sum()
    if n > 0:
        print(f"    {code:10s}  {n}")
print()

if len(df_perov) > 0:
    df_perov_ok = df_perov[df_perov['success']].sort_values('delta_pct')
    print("Perovskite additivity results (sorted by residual):")
    display(df_perov_ok[['compound', 'space_group', 'Z', 'maple_ternary',
                          'maple_binary_sum', 'delta_pct', 'classification',
                          'classification_label', 'madelung_factor']])



## 7. Pyrochlore-Specific Analysis

Filter for pyrochlore space groups (Fd-3m ordered, Fm-3m disordered).

In [ ]:
# 7. Pyrochlore-Specific Analysis
# ─────────────────────────────────────────────────────────────────────────────

PYROCHLORE_SGS = ['Fd-3m', 'Fm-3m']

df_pyro = df_classified[
    df_classified['space_group'].isin(PYROCHLORE_SGS)
].copy()

print(f"Fd-3m / Fm-3m candidates: {len(df_pyro)} structures")
print(f"  Space groups: {df_pyro['space_group'].value_counts().to_dict()}")
print(f"  Successfully decomposed: {df_pyro['success'].sum()}")

# Classification breakdown for pyrochlores
print(f"\n  Classification breakdown:")
for code in CODE_ORDER:
    n = (df_pyro['classification'] == code).sum()
    if n > 0:
        print(f"    {code:10s}  {n}")
print()

if len(df_pyro) > 0:
    df_pyro_ok = df_pyro[df_pyro['success']].sort_values('delta_pct')
    print("Pyrochlore/fluorite additivity results:")
    display(df_pyro_ok[['compound', 'space_group', 'Z', 'maple_ternary',
                         'maple_binary_sum', 'delta_pct', 'classification',
                         'classification_label', 'disordered',
                         'madelung_factor']])

    # ── Look for specific composition series ──────────────────────────
    ti_pyro = df_pyro_ok[df_pyro_ok['compound'].str.contains('Ti2',
                                                              na=False)]
    if len(ti_pyro) > 1:
        print(f"\nRE2Ti2O7 series ({len(ti_pyro)} entries):")
        display(ti_pyro[['compound', 'space_group', 'maple_ternary',
                          'delta_pct', 'madelung_factor', 'disordered',
                          'classification']])

    zr_pyro = df_pyro_ok[df_pyro_ok['compound'].str.contains('Zr',
                                                              na=False)]
    if len(zr_pyro) > 1:
        print(f"\nRE2Zr2O7 series ({len(zr_pyro)} entries):")
        display(zr_pyro[['compound', 'space_group', 'maple_ternary',
                          'delta_pct', 'madelung_factor', 'disordered',
                          'classification']])


## 8. Export Results

In [ ]:
# 8. Export Results
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR = MAPLE_DIR

# Full classified table
out_csv = OUTPUT_DIR / 'additivity_classified.csv'
df_classified.to_csv(out_csv, index=False)
print(f"Saved: {out_csv}  ({len(df_classified)} rows)")

# Verify the new columns are present in the export
expected_cols = ['classification', 'classification_tier', 'classification_label']
missing_cols = [c for c in expected_cols if c not in df_classified.columns]
if missing_cols:
    print(f"  WARNING: missing columns in export: {missing_cols}")
else:
    print(f"  Columns: {expected_cols} — all present")

# Perovskite subset
if len(df_perov) > 0:
    perov_csv = OUTPUT_DIR / 'additivity_perovskites.csv'
    df_perov.to_csv(perov_csv, index=False)
    print(f"Saved: {perov_csv}  ({len(df_perov)} rows)")

# Pyrochlore subset
if len(df_pyro) > 0:
    pyro_csv = OUTPUT_DIR / 'additivity_pyrochlores.csv'
    df_pyro.to_csv(pyro_csv, index=False)
    print(f"Saved: {pyro_csv}  ({len(df_pyro)} rows)")

# Binary reference table
binary_csv = OUTPUT_DIR / 'binary_oxide_reference.csv'
df_binary.to_csv(binary_csv, index=False)
print(f"Saved: {binary_csv}  ({len(df_binary)} rows)")

# Missing binaries summary
if df_additivity[~df_additivity['success']]['missing_binaries'].any():
    missing_series = (df_additivity[~df_additivity['success']]
                      ['missing_binaries']
                      .str.split('; ')
                      .explode()
                      .value_counts())
    print(f"\nMissing binary references (add these to the table):")
    print(missing_series.head(20))

# Classification summary for the export log
print(f"\nExport classification summary:")
print(f"  Total structures: {len(df_classified)}")
for tier_num, tier_name in {0: 'Additive', 1: 'Tier 1', 2: 'Tier 2',
                             3: 'Tier 3'}.items():
    n = (df_classified['classification_tier'] == tier_num).sum()
    if n > 0:
        print(f"  {tier_name:12s}  {n}")

print("\nDone.")


## 9. Notes and Limitations

### Binary reference MAPLE values

Hopper reads the released table, `binary_oxide_maple_reference.json`;
the inline table is its fallback, and the two are checked against each
other. 23 values are estimates with no source, and the source recorded
for Cu$_2$O does not reproduce. They are kept so that results can be
reproduced, and every decomposition that uses one names it
(`estimated_binaries`, v1.2.9). Replace them as sources become available.

### Non-integer oxidation states

The decomposition function rounds oxidation states to the nearest
integer to determine the binary formula.  This is appropriate for
stoichiometric compounds but introduces ambiguity for mixed-valence
systems (*e.g.* Mn$^{3.5+}$ giving a mean that rounds to either
Mn$_2$O$_3$ or MnO$_2$).

### Oxygen non-stoichiometry

Hoppe’s additivity theorem applies strictly to stoichiometric
compounds.  Oxygen-deficient or oxygen-excess structures
(code 2.4) require modified decomposition that accounts
for vacancy or interstitial contributions.

### Z = 1 problem

Code 2.2 failures arise wherever *Z* is taken from compositional
reduction. MaplePy has derived *Z* from the site multiplicities since
v1.1, so 2.2 cannot arise from a MaplePy run; the code is kept for
output from other calculators.

In [ ]:
# 10. Additional Paper Figures
# ─────────────────────────────────────────────────────────────────────────────
#
# Figure 4: RE₂Ti₂O₇ titanate pyrochlore series
# Figure 5: RE₂Sn₂O₇ stannate pyrochlore series
# Figure 6: Lanthanide contraction in RE₂O₃ binary reference table
# Figure 7: Classification waterfall
#
# All figures use the Sheffield colour palette and style conventions
# established in earlier cells.
# ─────────────────────────────────────────────────────────────────────────────

import re as regex

# Shannon ionic radii (Å), CN = 8, oxidation state +3
# (pyrochlore A-site is 8-coordinate)
SHANNON_CN8 = {
    'La': 1.160, 'Ce': 1.143, 'Pr': 1.126, 'Nd': 1.109, 'Pm': 1.093,
    'Sm': 1.079, 'Eu': 1.066, 'Gd': 1.053, 'Tb': 1.040, 'Dy': 1.027,
    'Ho': 1.015, 'Er': 1.004, 'Tm': 0.994, 'Yb': 0.985, 'Lu': 0.977,
    'Y':  1.019,
}

# Shannon ionic radii (Å), CN = 6, oxidation state +3
# (used for binary RE₂O₃ figure)
SHANNON_CN6 = {
    'La': 1.032, 'Ce': 1.01,  'Pr': 0.99,  'Nd': 0.983, 'Pm': 0.97,
    'Sm': 0.958, 'Eu': 0.947, 'Gd': 0.938, 'Tb': 0.923, 'Dy': 0.912,
    'Ho': 0.901, 'Er': 0.89,  'Tm': 0.88,  'Yb': 0.868, 'Lu': 0.861,
}


def _build_pyrochlore_series(df_classified, pattern, shannon=SHANNON_CN8):
    """
    Extract median Δ and MF per RE element for a pyrochlore series.

    Uses only Additive + 3.1 entries (excludes disorder, Z=1, oxi-state
    artefacts).  Returns a DataFrame sorted by decreasing ionic radius.
    """
    rows = []
    for comp in sorted(df_classified['compound'].unique()):
        m = regex.match(pattern, comp)
        if not m:
            continue
        re_el = m.group(1)
        if re_el not in shannon:
            continue

        clean = df_classified[
            (df_classified['compound'] == comp) &
            (df_classified['classification'].isin(['Additive', '3.1']))
        ]
        if len(clean) == 0:
            continue

        all_entries = df_classified[df_classified['compound'] == comp]
        radius = shannon[re_el]

        rows.append({
            'element': re_el, 'compound': comp, 'radius': radius,
            'delta_median': clean['delta_pct'].median(),
            'delta_mean':   clean['delta_pct'].mean(),
            'delta_std':    clean['delta_pct'].std() if len(clean) > 1 else 0,
            'delta_min':    clean['delta_pct'].min(),
            'delta_max':    clean['delta_pct'].max(),
            'mf_median':    clean['madelung_factor'].median(),
            'n_clean':      len(clean),
            'n_total':      len(all_entries),
        })

    df = pd.DataFrame(rows)
    if len(df) == 0:
        return df
    return df.sort_values('radius', ascending=False)


def plot_pyrochlore_series(df_classified, pattern, series_label,
                           fig_label='(a)', save_path=None):
    """
    Two-panel figure for a pyrochlore RE series:
      (a) Δ vs ionic radius — median ± range, with 1% threshold line
      (b) Madelung factor vs ionic radius
    """
    df = _build_pyrochlore_series(df_classified, pattern)
    if len(df) == 0:
        print(f"No data found for pattern {pattern}")
        return

    fig, (ax_a, ax_b) = plt.subplots(
        1, 2, figsize=(12, 5),
        gridspec_kw={'width_ratios': [3, 2], 'wspace': 0.30}
    )

    radii = df['radius'].values
    delta_med = df['delta_median'].values
    delta_min = df['delta_min'].values
    delta_max = df['delta_max'].values
    mf_med = df['mf_median'].values
    elements = df['element'].values
    n_clean = df['n_clean'].values

    # ── Panel (a): Δ vs ionic radius ─────────────────────────────────
    # Error bars showing full range
    yerr_lo = delta_med - delta_min
    yerr_hi = delta_max - delta_med
    ax_a.errorbar(radii, delta_med, yerr=[yerr_lo, yerr_hi],
                  fmt='o', color=SHEFFIELD['brand_violet'],
                  markerfacecolor=SHEFFIELD['brand_violet'],
                  markeredgecolor='white', markersize=8,
                  ecolor=SHEFFIELD['powder_violet'], elinewidth=1.2,
                  capsize=3, capthick=1.0, zorder=3)

    # 1% threshold line
    ax_a.axhline(y=1.0, color=SHEFFIELD['coral'], linestyle='--',
                 linewidth=1.2, alpha=0.7, zorder=1)
    ax_a.text(ax_a.get_xlim()[0] + 0.003, 1.05, 'Hoppe 1%',
              fontsize=9, color=SHEFFIELD['coral'], va='bottom')

    # Label each point
    for el, r, d, n in zip(elements, radii, delta_med, n_clean):
        n_str = f' ({n})' if n > 1 else ''
        ax_a.annotate(f'{el}{n_str}', (r, d),
                      textcoords='offset points', xytext=(0, 10),
                      ha='center', fontsize=8, color=SHEFFIELD['midnight'])

    ax_a.set_xlabel('Shannon ionic radius (CN = 8, 3+) / Å')
    ax_a.set_ylabel(r'Additivity residual, $\Delta$ / %')
    ax_a.set_title(f'{fig_label} {series_label}', loc='left', fontweight='bold')
    ax_a.spines['top'].set_visible(False)
    ax_a.spines['right'].set_visible(False)
    ax_a.invert_xaxis()

    # ── Panel (b): MF vs ionic radius ────────────────────────────────
    ax_b.plot(radii, mf_med, 'o-',
              color=SHEFFIELD['teal'], markerfacecolor=SHEFFIELD['teal'],
              markeredgecolor='white', markersize=8, linewidth=1.5, zorder=3)

    for el, r, mf in zip(elements, radii, mf_med):
        ax_b.annotate(el, (r, mf), textcoords='offset points',
                      xytext=(0, 10), ha='center', fontsize=8,
                      color=SHEFFIELD['midnight'])

    ax_b.set_xlabel('Shannon ionic radius (CN = 8, 3+) / Å')
    ax_b.set_ylabel('Madelung factor')
    ax_b.set_title('(b) Madelung factor', loc='left', fontweight='bold')
    ax_b.spines['top'].set_visible(False)
    ax_b.spines['right'].set_visible(False)
    ax_b.invert_xaxis()

    fig.tight_layout()

    if save_path:
        fig.savefig(save_path)
        print(f"Saved: {save_path}")

    plt.show()


def plot_lanthanide_contraction(binary_lookup, save_path=None):
    """
    Figure 6: Lanthanide contraction in RE₂O₃ binary MAPLE values.
    """
    lanthanides = ['La','Ce','Pr','Nd','Pm','Sm','Eu','Gd',
                   'Tb','Dy','Ho','Er','Tm','Yb','Lu']

    elements, radii, maples = [], [], []
    for el in lanthanides:
        formula = f"{el}2O3"
        if formula in binary_lookup and el in SHANNON_CN6:
            elements.append(el)
            radii.append(SHANNON_CN6[el])
            maples.append(binary_lookup[formula])

    if not elements:
        print("No RE₂O₃ entries found in binary reference table.")
        return

    fig, ax = plt.subplots(figsize=(8, 5))

    # La₂O₃ is P-3m1, rest are Ia-3
    ia3_idx = [i for i, el in enumerate(elements) if el != 'La']
    la_idx  = [i for i, el in enumerate(elements) if el == 'La']

    # Plot Ia-3 series
    ia3_r = [radii[i] for i in ia3_idx]
    ia3_m = [maples[i] for i in ia3_idx]
    ax.plot(ia3_r, ia3_m, 'o-',
            color=SHEFFIELD['brand_violet'], markerfacecolor=SHEFFIELD['brand_violet'],
            markeredgecolor='white', markersize=8, linewidth=1.5, zorder=3)

    # Linear fit to Ia-3 entries
    coeffs = np.polyfit(ia3_r, ia3_m, 1)
    fit_r = np.linspace(min(ia3_r) - 0.02, max(ia3_r) + 0.02, 100)
    fit_m = np.polyval(coeffs, fit_r)
    ax.plot(fit_r, fit_m, '--', color=SHEFFIELD['powder_violet'],
            linewidth=1.0, alpha=0.7, zorder=1)

    # Plot La separately (different structure type)
    if la_idx:
        i = la_idx[0]
        ax.plot(radii[i], maples[i], 's',
                color=SHEFFIELD['coral'], markeredgecolor='white',
                markersize=10, zorder=4)
        ax.annotate(r'$P\overline{3}m1$', (radii[i], maples[i]),
                    textcoords='offset points', xytext=(-20, -18),
                    fontsize=8, color=SHEFFIELD['coral'], fontstyle='italic',
                    arrowprops=dict(arrowstyle='-', color=SHEFFIELD['coral'],
                                    lw=0.8, alpha=0.6))

    # Label each point
    for el, r, m in zip(elements, radii, maples):
        idx = elements.index(el)
        va = 'bottom' if idx % 2 == 0 else 'top'
        offset = 12 if va == 'bottom' else -12
        ax.annotate(el, (r, m), textcoords='offset points',
                    xytext=(0, offset), ha='center', va=va,
                    fontsize=8, color=SHEFFIELD['midnight'])

    slope = coeffs[0]
    ax.text(0.03, 0.95,
            f'$Ia\\overline{{3}}$ fit: slope = {slope:.0f}'
            r' kcal mol$^{-1}$ Å$^{-1}$',
            transform=ax.transAxes, fontsize=9,
            color=SHEFFIELD['powder_violet'], va='top')

    ax.set_xlabel('Shannon ionic radius (CN = 6, 3+) / Å')
    ax.set_ylabel(r'MAPLE(RE$_2$O$_3$) / kcal mol$^{-1}$')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.invert_xaxis()

    fig.tight_layout()

    if save_path:
        fig.savefig(save_path)
        print(f"Saved: {save_path}")

    plt.show()


def plot_classification_waterfall(df_classified, save_path=None):
    """
    Figure 7: Classification waterfall — horizontal stacked bar showing
    how structures are sorted into classification tiers.
    """
    code_order = CODE_ORDER
    # v1.2.7: was a hand-written dict of seven codes, which raised
    # KeyError on the first structure carrying any of the other five.
    code_labels = CLASSIFICATION_LABELS
    # v1.2.7: was a hand-written dict of seven codes, which raised
    # KeyError on the first structure carrying any of the other five.
    code_colours = CLASSIFICATION_COLOURS

    counts = []
    codes_present = []
    for code in code_order:
        n = (df_classified['classification'] == code).sum()
        if n > 0:
            counts.append(n)
            codes_present.append(code)

    total = sum(counts)

    # ── Plan the code labels before the figure exists ───────────────────
    #
    # v1.2.8: rows used to alternate on index parity, which overprints as soon
    # as three narrow slices are adjacent. A label now takes the highest row
    # where it clears the label already there, measured from its own width,
    # and drops a row only when it has to. Planning before the figure is made
    # lets the figure be tall enough for however many rows that needs, rather
    # than clipping them into a fixed height.
    #
    # CHAR_W is one character at fontsize 8 in data units: 8 pt is about
    # 4.8 pt wide, 0.067 in, and the axis is 10 in across spanning `total`.
    # ROW_H must exceed the drawn text height or rows collide even when the
    # horizontal test passes, which is exactly how the first attempt failed.
    CHAR_W = total * 0.0067
    PAD    = total * 0.008
    ROW_H  = 0.24

    _rows, _plan = [], []
    left = 0
    for code, count in zip(codes_present, counts):
        cx = left + count / 2
        label = 'Add.' if code == 'Additive' else code
        if count < 10:             # narrow: carry the count in the label
            label = f'{label} ({count})'
        half_w = CHAR_W * len(label) / 2
        row = 0
        while row < len(_rows) and cx - _rows[row][0] < half_w + _rows[row][1] + PAD:
            row += 1
        if row == len(_rows):
            _rows.append((cx, half_w))
        else:
            _rows[row] = (cx, half_w)
        _plan.append((label, cx, row))
        left += count
    n_label_rows = len(_rows)

    fig, ax = plt.subplots(
        figsize=(10, 3.5 + 0.45 * max(0, n_label_rows - 2)))

    left = 0
    for code, count in zip(codes_present, counts):
        ax.barh(0, count, left=left, height=0.6,
                color=code_colours[code],
                edgecolor='white', linewidth=1.5)

        if count > 0:
            pct = count / total * 100
            cx = left + count / 2
            dark_codes = {'Additive', '2.2', '2.4', '1.1', '2.3'}
            text_colour = SHEFFIELD['midnight'] if code in dark_codes else 'white'

            if count >= 30:
                ax.text(cx, 0, f'{count}\n({pct:.0f}%)',
                        ha='center', va='center', fontsize=9,
                        fontweight='bold', color=text_colour)
            elif count >= 10:
                ax.text(cx, 0, f'{count}',
                        ha='center', va='center', fontsize=8,
                        fontweight='bold', color=text_colour)
            # v1.2.8: a narrow slice used to print its count above the bar,
            # where it collided with the tier brackets at y = 0.48. The count
            # goes into the label below instead, so each slice has exactly one
            # piece of text attached to it.

        left += count

    # ── Category labels below, one per slice ────────────────────────────
    for label, cx, row in _plan:
        ax.annotate(label, xy=(cx, -0.30), xytext=(cx, -0.38 - ROW_H * row),
                    ha='center', va='top', fontsize=8,
                    fontweight='bold', color=SHEFFIELD['midnight'],
                    arrowprops=dict(arrowstyle='-', color='grey',
                                    lw=0.6, alpha=0.5))

    # Tier brackets above
    tier_groups = {}
    left = 0
    for code, count in zip(codes_present, counts):
        tier = 0 if code == 'Additive' else int(code[0])
        tier_name = f'Tier {tier}'
        if tier_name not in tier_groups:
            tier_groups[tier_name] = [left, left + count]
        else:
            tier_groups[tier_name][1] = left + count
        left += count

    tier_colours = {
        'Tier 0': SHEFFIELD['peak_green'],
        'Tier 1': SHEFFIELD['powder_violet'],
        'Tier 2': SHEFFIELD['coral'],
        'Tier 3': SHEFFIELD['teal'],
    }

    for tier_name, (x0, x1) in tier_groups.items():
        width = x1 - x0
        if width > 15:
            mid = (x0 + x1) / 2
            colour = tier_colours.get(tier_name, SHEFFIELD['midnight'])
            ax.annotate('', xy=(x0, 0.48), xytext=(x1, 0.48),
                        arrowprops=dict(arrowstyle='|-|', color=colour,
                                        lw=1.2, mutation_scale=4))
            ax.text(mid, 0.58, tier_name, ha='center', va='bottom',
                    fontsize=8, color=colour, fontweight='bold')

    ax.set_xlim(-5, total + 5)
    # v1.2.8: deep enough for the rows the declutter actually used, so a
    # crowded dataset is not silently clipped.
    ax.set_ylim(min(-1.0, -0.55 - ROW_H * n_label_rows), 1.0)
    ax.set_xlabel(f'Number of structures (total = {total})')
    ax.set_yticks([])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)

    fig.tight_layout()

    if save_path:
        fig.savefig(save_path)
        print(f"Saved: {save_path}")

    plt.show()


# ── Generate Figures ─────────────────────────────────────────────────────────

# Figure 4: Titanate pyrochlore series
plot_pyrochlore_series(
    df_classified,
    pattern=r'^([A-Z][a-z]?)2Ti2O7$',
    series_label=r'RE$_2$Ti$_2$O$_7$ titanate series',
    fig_label='(a)',
    save_path=MAPLE_DIR / 'fig4_titanate_series.png'
)

# Figure 5: Stannate pyrochlore series
plot_pyrochlore_series(
    df_classified,
    pattern=r'^([A-Z][a-z]?)2Sn2O7$',
    series_label=r'RE$_2$Sn$_2$O$_7$ stannate series',
    fig_label='(a)',
    save_path=MAPLE_DIR / 'fig5_stannate_series.png'
)

# Figure 6: Lanthanide contraction in binary RE₂O₃
binary_lookup_fig6 = dict(zip(df_binary['formula'], df_binary['maple_kcal_mol']))
plot_lanthanide_contraction(
    binary_lookup_fig6,
    save_path=MAPLE_DIR / 'fig6_lanthanide_contraction.png'
)

# Figure 7: Classification waterfall
plot_classification_waterfall(
    df_classified,
    save_path=MAPLE_DIR / 'fig7_classification_waterfall.png'
)